# C1_technical_indicators: Classical technical indicators

Public appendix notebook from the Bachelor's Thesis *Evaluación robusta de estrategias de inversión: backtesting, sobreajuste y validación estadística*.

This notebook is included for transparency/reproducibility. It was originally developed in Google Colab; paths may need to be adapted for local execution.


In [ ]:
# -*- coding: utf-8 -*-
"""block1.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1nUrbCOLKHQ2wWCugKRA0mPcFG-aucRPP

Instalación
"""

!pip -q install pandas numpy yfinance matplotlib lxml html5lib beautifulsoup4 requests


In [ ]:
# ============================================================
# CELDA 1 - MONTAR GOOGLE DRIVE Y CREAR CARPETAS


In [ ]:
# ============================================================

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# Carpeta base del TFG en Drive
TFG_BASE_DIR = "/content/drive/MyDrive/TFG"

# Crear carpetas necesarias
Path(f"{TFG_BASE_DIR}").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/data").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/data/cache").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/results").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/logs").mkdir(parents=True, exist_ok=True)

print("Google Drive montado.")
print(f"Carpeta base del TFG: {TFG_BASE_DIR}")
print("Carpetas creadas correctamente.")

"""Código completo del Bloque 0"""


In [ ]:
# ============================================================
# CAP 0 - MOTOR REPRODUCIBLE DE DATOS, BACKTESTING Y MÉTRICAS
# Versión Colab con reconstrucción histórica del S&P 100


In [ ]:
# ============================================================

from __future__ import annotations

import json
import math
import time
import warnings
import re
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path
from typing import Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import requests


In [ ]:
# ============================================================
# 1. CONFIGURACIÓN GENERAL


In [ ]:
# ============================================================

@dataclass
class BacktestConfig:
    # Periodo de análisis.
    # Yahoo Finance usa end_date como fecha final EXCLUSIVA.
    start_date: str = "2021-01-01"
    end_date: str = "2026-01-01"

    # Datos
    data_source: str = "Yahoo Finance via yfinance"
    universe_name: str = "S&P 100"

    # Archivo manual si algún día tenemos una fuente histórica propia.
    membership_file: str = "data/sp100_membership.csv"

    # Archivo reconstruido automáticamente desde revisiones históricas de Wikipedia.
    reconstructed_membership_file: str = "data/sp100_membership_reconstructed_wikipedia.csv"
    snapshot_log_file: str = "data/sp100_wikipedia_snapshot_log.csv"

    # Reconstrucción histórica
    use_reconstructed_membership: bool = True
    force_rebuild_membership: bool = True
    membership_history_frequency: str = "MS"

    # Importante:
    # Si esto está en False, el código NO vuelve a una lista actual si falla la reconstrucción.
    # Así evitamos meter survivorship bias sin querer.
    allow_current_fallback: bool = False

    # Benchmark
    # SPY = proxy invertible del S&P 500 con precios ajustados.
    # ^GSPC = índice S&P 500 de precio, no total return.
    benchmark_ticker: str = "SPY"
    benchmark_name: str = "S&P 500 benchmark"

    # Backtest
    initial_capital: float = 1.0
    trading_days_per_year: int = 252

    # Costes
    transaction_cost: float = 0.001
    cost_sensitivity: Tuple[float, ...] = (0.0, 0.0005, 0.001, 0.0025)

    # Tasa libre de riesgo anual provisional.
    risk_free_rate_annual: float = 0.0225

    # Limpieza
    min_price_coverage: float = 0.80
    min_assets_required: int = 20

    # Estrategia
    # "static_buy_hold": compra al inicio el universo histórico inicial y no rebalancea.
    # "dynamic_equal_weight": usa el universo histórico correspondiente en cada rebalanceo.
    strategy_mode: str = "dynamic_equal_weight"
    rebalance_frequency: str = "M"

    # Reproducibilidad
    random_seed: int = 123
    output_dir: str = "results/block0_sp100_survivorship_corrected"


CONFIG = BacktestConfig()


In [ ]:
# ============================================================
# 2. UTILIDADES GENERALES


In [ ]:
# ============================================================

def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def now_utc_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def save_json(obj: dict, path: str | Path) -> None:
    path = Path(path)
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=4, ensure_ascii=False, default=str)


def to_yfinance_ticker(ticker: str) -> str:
    """
    Yahoo Finance usa '-' en vez de '.' para clases de acciones.
    Ejemplo: BRK.B -> BRK-B.
    """
    return str(ticker).strip().replace(".", "-")


def from_yfinance_ticker(ticker: str) -> str:
    return str(ticker).strip().replace("-", ".")


In [ ]:
# ============================================================
# 3. UNIVERSO S&P 100 HISTÓRICO


In [ ]:
# ============================================================

WIKIPEDIA_API_URL = "https://en.wikipedia.org/w/api.php"
WIKIPEDIA_TITLE_SP100 = "S&P 100"


# Lista solo de emergencia.
# Por defecto NO se usa porque allow_current_fallback=False.
SP100_FALLBACK_TICKERS = [
    "AAPL", "ABBV", "ABT", "ACN", "ADBE", "AIG", "AMD", "AMGN", "AMT", "AMZN",
    "AVGO", "AXP", "BA", "BAC", "BK", "BKNG", "BLK", "BMY", "BRK.B", "C",
    "CAT", "CHTR", "CL", "CMCSA", "COF", "COP", "COST", "CRM", "CSCO", "CVS",
    "CVX", "DE", "DHR", "DIS", "DUK", "EMR", "EXC", "F", "FDX", "GD",
    "GE", "GILD", "GM", "GOOG", "GOOGL", "GS", "HD", "HON", "IBM", "INTC",
    "JNJ", "JPM", "KHC", "KO", "LIN", "LLY", "LMT", "LOW", "MA", "MCD",
    "MDLZ", "MDT", "MET", "META", "MMM", "MO", "MRK", "MS", "MSFT", "NEE",
    "NFLX", "NKE", "NVDA", "ORCL", "PEP", "PFE", "PG", "PM", "PYPL", "QCOM",
    "RTX", "SBUX", "SCHW", "SO", "SPG", "T", "TGT", "TMO", "TXN", "UNH",
    "UNP", "UPS", "USB", "V", "VZ", "WBA", "WFC", "WMT", "XOM"
]


def clean_ticker_symbol(x: str) -> Optional[str]:
    """
    Limpia símbolos extraídos de tablas HTML.
    """
    if pd.isna(x):
        return None

    s = str(x).strip()
    s = re.sub(r"\[.*?\]", "", s)
    s = s.replace("\xa0", " ")
    s = s.strip()

    if not s:
        return None

    bad_values = {"symbol", "ticker", "nan", "none", "company", "security"}
    if s.lower() in bad_values:
        return None

    s = re.sub(r"[^A-Za-z0-9\.\-]", "", s)

    if not s:
        return None

    return s.upper()


def wikipedia_api_get(
    params: dict,
    max_retries: int = 4,
    sleep_seconds: float = 0.8,
) -> dict:
    """
    Petición robusta a la API de Wikipedia.
    """
    headers = {
        "User-Agent": "TFG-Backtest-Research/1.0 academic Colab notebook"
    }

    last_error = None

    for attempt in range(max_retries):
        try:
            response = requests.get(
                WIKIPEDIA_API_URL,
                params=params,
                headers=headers,
                timeout=30,
            )

            response.raise_for_status()
            data = response.json()

            if "error" in data:
                raise RuntimeError(data["error"])

            return data

        except Exception as e:
            last_error = e
            time.sleep(sleep_seconds * (attempt + 1))

    raise RuntimeError(f"No se pudo consultar Wikipedia API. Último error: {last_error}")


def get_wikipedia_revision_at(date: pd.Timestamp) -> Tuple[int, str]:
    """
    Obtiene la revisión de la página S&P 100 existente en una fecha concreta.
    """
    date = pd.Timestamp(date)
    rvstart = date.strftime("%Y-%m-%dT23:59:59Z")

    params = {
        "action": "query",
        "format": "json",
        "formatversion": "2",
        "prop": "revisions",
        "titles": WIKIPEDIA_TITLE_SP100,
        "rvprop": "ids|timestamp",
        "rvlimit": "1",
        "rvdir": "older",
        "rvstart": rvstart,
    }

    data = wikipedia_api_get(params)
    pages = data.get("query", {}).get("pages", [])

    if not pages or "revisions" not in pages[0] or not pages[0]["revisions"]:
        raise RuntimeError(f"No se encontró revisión de Wikipedia para {date.date()}")

    rev = pages[0]["revisions"][0]

    return int(rev["revid"]), str(rev["timestamp"])


def get_wikipedia_revision_html(revid: int) -> str:
    """
    Convierte una revisión concreta a HTML mediante action=parse.
    """
    params = {
        "action": "parse",
        "format": "json",
        "oldid": str(revid),
        "prop": "text",
    }

    data = wikipedia_api_get(params)
    html = data.get("parse", {}).get("text", {}).get("*")

    if not html:
        raise RuntimeError(f"No se pudo extraer HTML para oldid={revid}")

    return html


def extract_sp100_tickers_from_html(html: str) -> List[str]:
    """
    Extrae los tickers de la tabla de constituyentes del S&P 100.

    Busca tablas con columna Symbol/Ticker.
    Acepta una tabla si encuentra aproximadamente entre 80 y 120 símbolos.
    """
    tables = pd.read_html(StringIO(html))
    candidates = []

    for table in tables:
        table = table.copy()

        if isinstance(table.columns, pd.MultiIndex):
            table.columns = [
                " ".join([str(x) for x in col if str(x) != "nan"]).strip()
                for col in table.columns
            ]
        else:
            table.columns = [str(c).strip() for c in table.columns]

        symbol_col = None

        for col in table.columns:
            col_lower = str(col).lower()
            if "symbol" in col_lower or "ticker" in col_lower:
                symbol_col = col
                break

        if symbol_col is None:
            continue

        tickers = []

        for value in table[symbol_col].tolist():
            ticker = clean_ticker_symbol(value)
            if ticker is not None:
                tickers.append(ticker)

        tickers = sorted(set(tickers))

        if 80 <= len(tickers) <= 120:
            candidates.append(tickers)

    if not candidates:
        raise RuntimeError("No se encontró una tabla válida de constituyentes del S&P 100.")

    candidates = sorted(candidates, key=lambda x: abs(len(x) - 100))

    return candidates[0]


def fetch_sp100_snapshot_from_wikipedia(date: pd.Timestamp) -> Tuple[List[str], dict]:
    """
    Descarga la composición del S&P 100 según la página de Wikipedia
    tal como estaba en la fecha indicada.
    """
    revid, revision_timestamp = get_wikipedia_revision_at(date)
    html = get_wikipedia_revision_html(revid)
    tickers = extract_sp100_tickers_from_html(html)

    metadata = {
        "snapshot_date": str(pd.Timestamp(date).date()),
        "wiki_revision_id": revid,
        "wiki_revision_timestamp": revision_timestamp,
        "n_tickers": len(tickers),
        "status": "ok",
    }

    return tickers, metadata


def make_snapshot_dates(start_date: str, end_date: str, freq: str) -> List[pd.Timestamp]:
    """
    Genera fechas de snapshot entre start_date y end_date.

    end_date se considera exclusiva, igual que en Yahoo Finance.
    """
    start = pd.Timestamp(start_date).normalize()
    end_exclusive = pd.Timestamp(end_date).normalize()
    end_inclusive = end_exclusive - pd.Timedelta(days=1)

    regular_dates = pd.date_range(start=start, end=end_inclusive, freq=freq)

    dates = sorted(set([start] + list(regular_dates)))
    dates = [pd.Timestamp(d).normalize() for d in dates if start <= d <= end_inclusive]

    return dates


def snapshots_to_membership(snapshot_rows: List[dict]) -> pd.DataFrame:
    """
    Convierte snapshots mensuales en intervalos de pertenencia.
    """
    snapshots = pd.DataFrame(snapshot_rows)

    if snapshots.empty:
        raise RuntimeError("No hay snapshots para construir membership.")

    snapshots["snapshot_date"] = pd.to_datetime(snapshots["snapshot_date"])
    snapshots = snapshots.sort_values("snapshot_date")

    interval_rows = []
    unique_dates = sorted(snapshots["snapshot_date"].unique())

    for i, date in enumerate(unique_dates):
        current = snapshots[snapshots["snapshot_date"] == date]
        tickers = sorted(current["ticker"].dropna().unique().tolist())

        if i < len(unique_dates) - 1:
            interval_end = pd.Timestamp(unique_dates[i + 1]) - pd.Timedelta(days=1)
        else:
            interval_end = pd.NaT

        for ticker in tickers:
            interval_rows.append({
                "ticker": ticker,
                "yf_ticker": to_yfinance_ticker(ticker),
                "start": pd.Timestamp(date),
                "end": interval_end,
                "source": "wikipedia_historical_revision_snapshot",
            })

    membership = pd.DataFrame(interval_rows)

    if membership.empty:
        raise RuntimeError("Membership histórico vacío.")

    return membership


def reconstruct_sp100_membership_from_wikipedia(
    config: BacktestConfig,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Reconstruye la composición histórica del S&P 100 usando revisiones históricas
    de Wikipedia.

    Esto evita usar una lista actual para todo el pasado.
    """
    snapshot_dates = make_snapshot_dates(
        config.start_date,
        config.end_date,
        config.membership_history_frequency,
    )

    print("Reconstruyendo S&P 100 histórico desde Wikipedia.")
    print(f"Snapshots: {len(snapshot_dates)} fechas ({config.membership_history_frequency})")

    snapshot_rows = []
    snapshot_log = []

    last_good_tickers = None

    for i, date in enumerate(snapshot_dates):
        print(f"  Snapshot {i + 1}/{len(snapshot_dates)}: {date.date()}")

        try:
            tickers, meta = fetch_sp100_snapshot_from_wikipedia(date)
            last_good_tickers = tickers

        except Exception as e:
            if last_good_tickers is None:
                raise RuntimeError(
                    f"Falló el primer snapshot ({date.date()}) y no hay composición anterior para arrastrar. "
                    f"Error: {e}"
                )

            tickers = last_good_tickers
            meta = {
                "snapshot_date": str(date.date()),
                "wiki_revision_id": None,
                "wiki_revision_timestamp": None,
                "n_tickers": len(tickers),
                "status": f"carried_forward_after_error: {type(e).__name__}: {e}",
            }

        for ticker in tickers:
            snapshot_rows.append({
                "snapshot_date": str(date.date()),
                "ticker": ticker,
            })

        snapshot_log.append(meta)

        time.sleep(0.25)

    membership = snapshots_to_membership(snapshot_rows)
    snapshot_log_df = pd.DataFrame(snapshot_log)

    ensure_dir("data")

    membership.to_csv(config.reconstructed_membership_file, index=False)
    snapshot_log_df.to_csv(config.snapshot_log_file, index=False)

    return membership, snapshot_log_df


def load_membership_schedule(config: BacktestConfig) -> Tuple[pd.DataFrame, str]:
    """
    Carga o reconstruye el membership histórico.

    Orden:
    1. Si existe data/sp100_membership.csv manual, lo usa.
    2. Si existe membership reconstruido y force_rebuild_membership=False, lo usa.
    3. Si no existe, reconstruye desde revisiones históricas de Wikipedia.
    4. Si todo falla y allow_current_fallback=False, lanza error.
    """
    manual_path = Path(config.membership_file)
    reconstructed_path = Path(config.reconstructed_membership_file)

    if manual_path.exists():
        membership = pd.read_csv(manual_path)

        required = {"ticker", "start", "end"}
        missing = required - set(membership.columns)

        if missing:
            raise ValueError(f"Faltan columnas en {manual_path}: {missing}")

        membership["ticker"] = membership["ticker"].astype(str).str.strip()
        membership["yf_ticker"] = membership["ticker"].map(to_yfinance_ticker)
        membership["start"] = pd.to_datetime(membership["start"], errors="coerce")
        membership["end"] = pd.to_datetime(membership["end"], errors="coerce")

        return membership, "manual_historical_membership_file"

    if (
        config.use_reconstructed_membership
        and reconstructed_path.exists()
        and not config.force_rebuild_membership
    ):
        membership = pd.read_csv(reconstructed_path)

        membership["ticker"] = membership["ticker"].astype(str).str.strip()
        membership["yf_ticker"] = membership["ticker"].map(to_yfinance_ticker)
        membership["start"] = pd.to_datetime(membership["start"], errors="coerce")
        membership["end"] = pd.to_datetime(membership["end"], errors="coerce")

        return membership, "cached_wikipedia_reconstructed_historical_membership"

    if config.use_reconstructed_membership:
        try:
            membership, snapshot_log_df = reconstruct_sp100_membership_from_wikipedia(config)
            return membership, "wikipedia_reconstructed_historical_membership"

        except Exception as e:
            if not config.allow_current_fallback:
                raise RuntimeError(
                    "No se pudo reconstruir la composición histórica del S&P 100 y "
                    "allow_current_fallback=False. No se usará lista actual para evitar survivorship bias. "
                    f"Error original: {e}"
                )

    if config.allow_current_fallback:
        tickers = sorted(SP100_FALLBACK_TICKERS)

        membership = pd.DataFrame({
            "ticker": tickers,
            "yf_ticker": [to_yfinance_ticker(t) for t in tickers],
            "start": pd.Timestamp("1900-01-01"),
            "end": pd.NaT,
            "source": "current_fallback",
        })

        return membership, "current_fallback_SURVIVORSHIP_BIASED"

    raise RuntimeError("No se pudo cargar ni reconstruir membership histórico.")


def active_tickers_on_date(membership: pd.DataFrame, date: pd.Timestamp) -> List[str]:
    """
    Devuelve los tickers activos en una fecha concreta según la tabla de pertenencia.
    """
    date = pd.Timestamp(date)

    active = membership[
        (membership["start"].isna() | (membership["start"] <= date)) &
        (membership["end"].isna() | (membership["end"] >= date))
    ]

    return sorted(active["yf_ticker"].dropna().unique().tolist())


def all_tickers_needed(membership: pd.DataFrame, start_date: str, end_date: str) -> List[str]:
    """
    Devuelve todos los tickers que podrían ser necesarios entre start_date y end_date.
    """
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)

    relevant = membership[
        (membership["end"].isna() | (membership["end"] >= start)) &
        (membership["start"].isna() | (membership["start"] <= end))
    ]

    return sorted(relevant["yf_ticker"].dropna().unique().tolist())


In [ ]:
# ============================================================
# 4. DESCARGA Y PREPROCESADO DE DATOS


In [ ]:
# ============================================================

def download_adjusted_close(
    tickers: Iterable[str],
    start_date: str,
    end_date: str,
    chunk_size: int = 25,
    sleep_seconds: float = 1.0,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Descarga precios ajustados mediante yfinance.

    Con auto_adjust=True, la columna Close queda ajustada.
    Descarga por bloques para reducir errores en Colab/Yahoo.
    """
    tickers = sorted(set([to_yfinance_ticker(t) for t in tickers]))

    if not tickers:
        raise ValueError("No hay tickers para descargar.")

    all_prices = []
    report_rows = []

    print(f"Descargando {len(tickers)} tickers desde Yahoo Finance...")

    for i in range(0, len(tickers), chunk_size):
        chunk = tickers[i:i + chunk_size]
        print(f"  Bloque {i // chunk_size + 1}: {len(chunk)} tickers")

        try:
            data = yf.download(
                tickers=chunk,
                start=start_date,
                end=end_date,
                auto_adjust=True,
                progress=False,
                group_by="ticker",
                threads=True,
            )

        except Exception as e:
            for t in chunk:
                report_rows.append({
                    "yf_ticker": t,
                    "status": f"download_exception: {type(e).__name__}",
                    "coverage": 0.0,
                    "first_valid_date": None,
                    "last_valid_date": None,
                })

            time.sleep(sleep_seconds)
            continue

        prices_chunk = pd.DataFrame()

        if len(chunk) == 1:
            t = chunk[0]

            if isinstance(data, pd.DataFrame) and "Close" in data.columns:
                prices_chunk[t] = data["Close"]

        else:
            for t in chunk:
                try:
                    if isinstance(data.columns, pd.MultiIndex):
                        if t in data.columns.get_level_values(0):
                            close = data[t]["Close"].copy()
                            prices_chunk[t] = close
                    else:
                        if "Close" in data.columns:
                            prices_chunk[t] = data["Close"]

                except Exception:
                    pass

        if not prices_chunk.empty:
            prices_chunk = prices_chunk.sort_index()
            prices_chunk.index.name = "Date"
            prices_chunk = prices_chunk.dropna(axis=1, how="all")
            all_prices.append(prices_chunk)

        available = set(prices_chunk.columns) if not prices_chunk.empty else set()

        for t in chunk:
            if t in available:
                s = prices_chunk[t]
                coverage = float(s.notna().mean())
                first_date = s.first_valid_index()
                last_date = s.last_valid_index()
                status = "ok"
            else:
                coverage = 0.0
                first_date = None
                last_date = None
                status = "download_failed_or_no_close"

            report_rows.append({
                "yf_ticker": t,
                "status": status,
                "coverage": coverage,
                "first_valid_date": first_date,
                "last_valid_date": last_date,
            })

        time.sleep(sleep_seconds)

    if not all_prices:
        raise RuntimeError("No se han podido descargar precios. Revisa conexión, tickers o yfinance.")

    prices = pd.concat(all_prices, axis=1)
    prices = prices.loc[:, ~prices.columns.duplicated()]
    prices = prices.sort_index()
    prices.index.name = "Date"

    report = pd.DataFrame(report_rows)

    return prices, report


def clean_prices(
    prices: pd.DataFrame,
    min_coverage: float,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Limpieza conservadora.
    """
    prices = prices.copy()
    prices = prices[~prices.index.duplicated(keep="first")]
    prices = prices.sort_index()

    coverage = prices.notna().mean()

    keep = coverage[coverage >= min_coverage].index.tolist()
    excluded = coverage[coverage < min_coverage].index.tolist()

    exclusions = pd.DataFrame({
        "yf_ticker": excluded,
        "reason": f"coverage_below_{min_coverage:.0%}",
        "coverage": coverage.loc[excluded].values if len(excluded) > 0 else [],
    })

    clean = prices[keep].copy()
    clean = clean.ffill()

    return clean, exclusions


def compute_simple_returns(prices: pd.DataFrame) -> pd.DataFrame:
    returns = prices.pct_change(fill_method=None)
    returns = returns.replace([np.inf, -np.inf], np.nan)

    return returns


In [ ]:
# ============================================================
# 5. BACKTESTS BASE


In [ ]:
# ============================================================

def backtest_static_buy_hold(
    prices: pd.DataFrame,
    membership: pd.DataFrame,
    config: BacktestConfig,
    transaction_cost: Optional[float] = None,
) -> Tuple[pd.Series, pd.Series, pd.DataFrame, dict]:
    """
    Buy & Hold estático:
    - Toma los miembros activos al inicio del backtest.
    - Compra cartera equally weighted.
    - No rebalancea.
    """
    transaction_cost = config.transaction_cost if transaction_cost is None else transaction_cost

    prices = prices.copy().dropna(axis=0, how="all")
    first_date = prices.index[0]

    active = active_tickers_on_date(membership, first_date)
    active = [t for t in active if t in prices.columns]

    first_prices = prices.loc[first_date, active].dropna()
    active = first_prices.index.tolist()

    if len(active) < config.min_assets_required:
        raise RuntimeError(
            f"Activos disponibles insuficientes para Buy & Hold: {len(active)} "
            f"(mínimo {config.min_assets_required})."
        )

    n = len(active)
    gross_weights = pd.Series(1.0 / n, index=active)

    entry_turnover = gross_weights.abs().sum()
    initial_cost = transaction_cost * entry_turnover * config.initial_capital
    investable_capital = config.initial_capital - initial_cost

    shares = (investable_capital * gross_weights) / first_prices

    portfolio_values = prices[active].ffill().mul(shares, axis=1).sum(axis=1)
    equity = portfolio_values.rename("strategy_equity")

    equity.iloc[0] = investable_capital

    returns = equity.pct_change(fill_method=None)
    returns.iloc[0] = equity.iloc[0] / config.initial_capital - 1.0
    returns = returns.rename("strategy_returns")

    weights = pd.DataFrame(0.0, index=prices.index, columns=active)
    weights.loc[:, active] = gross_weights.values

    metadata = {
        "strategy": "static_buy_hold",
        "first_date": str(first_date.date()),
        "n_assets": n,
        "transaction_cost": transaction_cost,
        "entry_turnover": float(entry_turnover),
        "initial_cost": float(initial_cost),
        "active_tickers": active,
    }

    return returns, equity, weights, metadata


def backtest_dynamic_equal_weight(
    prices: pd.DataFrame,
    membership: pd.DataFrame,
    config: BacktestConfig,
    transaction_cost: Optional[float] = None,
) -> Tuple[pd.Series, pd.Series, pd.DataFrame, dict]:
    """
    Equal-weight dinámico:
    - En cada fecha de rebalanceo selecciona miembros activos según membership histórico.
    - Rebalancea a pesos iguales.
    """
    transaction_cost = config.transaction_cost if transaction_cost is None else transaction_cost

    prices = prices.copy().dropna(axis=0, how="all").ffill()
    returns_assets = compute_simple_returns(prices).fillna(0.0)

    dates = prices.index

    rebalance_dates_raw = prices.resample(config.rebalance_frequency).last().index
    rebalance_dates = []

    for d in rebalance_dates_raw:
        idx = prices.index.get_indexer([d], method="pad")[0]

        if idx >= 0:
            rebalance_dates.append(prices.index[idx])

    rebalance_dates = sorted(set([d for d in rebalance_dates if d in dates]))

    all_cols = prices.columns.tolist()

    weights = pd.DataFrame(0.0, index=dates, columns=all_cols)

    current_weights = pd.Series(0.0, index=all_cols)
    prev_weights = pd.Series(0.0, index=all_cols)

    equity = pd.Series(index=dates, dtype=float, name="strategy_equity")
    strategy_returns = pd.Series(index=dates, dtype=float, name="strategy_returns")

    capital = config.initial_capital
    rebalance_count = 0
    total_turnover = 0.0

    for i, date in enumerate(dates):
        cost = 0.0

        if i == 0 or date in rebalance_dates:
            active = active_tickers_on_date(membership, date)
            active = [t for t in active if t in all_cols and not pd.isna(prices.loc[date, t])]

            if len(active) >= config.min_assets_required:
                target_weights = pd.Series(0.0, index=all_cols)
                target_weights.loc[active] = 1.0 / len(active)
            else:
                target_weights = prev_weights.copy()

            turnover = float((target_weights - prev_weights).abs().sum())
            cost = transaction_cost * turnover

            total_turnover += turnover
            rebalance_count += 1

            current_weights = target_weights.copy()
            prev_weights = current_weights.copy()

        if i == 0:
            r = -cost
        else:
            r_gross = float(current_weights.fillna(0.0).dot(returns_assets.iloc[i].fillna(0.0)))
            r = r_gross - cost

        capital *= (1.0 + r)

        equity.loc[date] = capital
        strategy_returns.loc[date] = r
        weights.loc[date] = current_weights.values

    metadata = {
        "strategy": "dynamic_equal_weight",
        "rebalance_frequency": config.rebalance_frequency,
        "transaction_cost": transaction_cost,
        "rebalance_count": rebalance_count,
        "total_turnover": total_turnover,
        "average_turnover_per_rebalance": total_turnover / rebalance_count if rebalance_count else np.nan,
    }

    return strategy_returns, equity, weights, metadata


def backtest_benchmark(
    benchmark_prices: pd.Series,
    config: BacktestConfig,
) -> Tuple[pd.Series, pd.Series]:
    """
    Benchmark Buy & Hold.
    No aplicamos costes al benchmark por defecto.
    """
    px = benchmark_prices.dropna().copy()
    px = px / px.iloc[0] * config.initial_capital
    px.name = "benchmark_equity"

    ret = px.pct_change(fill_method=None)
    ret.iloc[0] = 0.0
    ret.name = "benchmark_returns"

    return ret, px


In [ ]:
# ============================================================
# 6. MÉTRICAS


In [ ]:
# ============================================================

def format_index_endpoint(x):
    """
    Convierte un índice a string de forma robusta.
    """
    if hasattr(x, "date"):
        return str(x.date())

    return str(x)


def max_drawdown(equity: pd.Series) -> float:
    equity = equity.dropna()

    if len(equity) == 0:
        return np.nan

    running_max = equity.cummax()
    dd = equity / running_max - 1.0

    return float(dd.min())


def drawdown_series(equity: pd.Series) -> pd.Series:
    equity = equity.dropna()

    if len(equity) == 0:
        return pd.Series(dtype=float, name="drawdown")

    running_max = equity.cummax()

    return (equity / running_max - 1.0).rename("drawdown")


def annualized_volatility(
    returns: pd.Series,
    trading_days: int = 252,
) -> float:
    returns = returns.dropna()

    if len(returns) < 2:
        return np.nan

    return float(returns.std(ddof=1) * math.sqrt(trading_days))


def cagr_from_equity(
    equity: pd.Series,
    initial_capital: float,
    trading_days: int = 252,
) -> float:
    equity = equity.dropna()

    if len(equity) < 2:
        return np.nan

    years = len(equity) / trading_days
    final_value = equity.iloc[-1]

    if final_value <= 0:
        return -1.0

    return float((final_value / initial_capital) ** (1.0 / years) - 1.0)


def compute_metrics(
    returns: pd.Series,
    equity: pd.Series,
    config: BacktestConfig,
    label: str,
) -> pd.Series:
    returns = returns.dropna()
    equity = equity.dropna()

    if len(returns) == 0 or len(equity) == 0:
        return pd.Series({
            "label": label,
            "start": None,
            "end": None,
            "n_observations": 0,
            "cumulative_return": np.nan,
            "CAGR": np.nan,
            "annualized_volatility": np.nan,
            "Sharpe": np.nan,
            "Sortino": np.nan,
            "max_drawdown": np.nan,
            "Calmar": np.nan,
            "hit_ratio": np.nan,
            "risk_free_rate_annual": config.risk_free_rate_annual,
        })

    rf_daily = (1.0 + config.risk_free_rate_annual) ** (1.0 / config.trading_days_per_year) - 1.0
    excess = returns - rf_daily

    cumulative_return = float(equity.iloc[-1] / config.initial_capital - 1.0)

    cagr = cagr_from_equity(
        equity,
        config.initial_capital,
        config.trading_days_per_year,
    )

    vol = annualized_volatility(
        returns,
        config.trading_days_per_year,
    )

    if len(excess) > 1 and excess.std(ddof=1) > 0:
        sharpe = float(
            excess.mean() / excess.std(ddof=1) * math.sqrt(config.trading_days_per_year)
        )
    else:
        sharpe = np.nan

    downside = excess[excess < 0]

    if len(downside) > 1 and downside.std(ddof=1) > 0:
        sortino = float(
            excess.mean() / downside.std(ddof=1) * math.sqrt(config.trading_days_per_year)
        )
    else:
        sortino = np.nan

    mdd = max_drawdown(equity)

    if not np.isnan(mdd) and mdd < 0:
        calmar = float(cagr / abs(mdd))
    else:
        calmar = np.nan

    hit_ratio = float((returns > 0).mean())

    return pd.Series({
        "label": label,
        "start": format_index_endpoint(equity.index[0]),
        "end": format_index_endpoint(equity.index[-1]),
        "n_observations": int(len(returns)),
        "cumulative_return": cumulative_return,
        "CAGR": cagr,
        "annualized_volatility": vol,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "max_drawdown": mdd,
        "Calmar": calmar,
        "hit_ratio": hit_ratio,
        "risk_free_rate_annual": config.risk_free_rate_annual,
    })


def compute_turnover(weights: pd.DataFrame) -> pd.Series:
    if weights.empty:
        return pd.Series(dtype=float, name="turnover")

    turnover = weights.diff().abs().sum(axis=1)
    turnover.iloc[0] = weights.iloc[0].abs().sum()

    return turnover.rename("turnover")


In [ ]:
# ============================================================
# 7. BOOTSTRAP SIMPLE


In [ ]:
# ============================================================

def bootstrap_metrics_simple(
    returns: pd.Series,
    config: BacktestConfig,
    n_bootstrap: int = 1000,
) -> pd.DataFrame:
    """
    Bootstrap simple sobre rentabilidades.
    Más adelante podemos sustituirlo por block bootstrap.
    """
    rng = np.random.default_rng(config.random_seed)
    r = returns.dropna().values

    rows = []

    for b in range(n_bootstrap):
        sample = rng.choice(r, size=len(r), replace=True)
        equity = pd.Series(config.initial_capital * np.cumprod(1.0 + sample))
        sample_returns = pd.Series(sample)

        m = compute_metrics(sample_returns, equity, config, label=f"bootstrap_{b}")
        rows.append(m)

    return pd.DataFrame(rows)


In [ ]:
# ============================================================
# 8. FIGURAS


In [ ]:
# ============================================================

def plot_equity_curves(
    strategy_equity: pd.Series,
    benchmark_equity: pd.Series,
    output_path: str | Path,
) -> None:
    plt.figure(figsize=(10, 6))
    strategy_equity.dropna().plot(label="Strategy")
    benchmark_equity.dropna().plot(label="Benchmark")
    plt.title("Curva de capital")
    plt.xlabel("Fecha")
    plt.ylabel("Capital normalizado")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()


def plot_drawdowns(
    strategy_equity: pd.Series,
    benchmark_equity: pd.Series,
    output_path: str | Path,
) -> None:
    plt.figure(figsize=(10, 6))
    drawdown_series(strategy_equity).plot(label="Strategy")
    drawdown_series(benchmark_equity).plot(label="Benchmark")
    plt.title("Drawdown")
    plt.xlabel("Fecha")
    plt.ylabel("Drawdown")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()


In [ ]:
# ============================================================
# 9. PIPELINE PRINCIPAL


In [ ]:
# ============================================================

def run_block0(config: BacktestConfig = CONFIG) -> None:
    warnings.filterwarnings("ignore", category=FutureWarning)
    np.random.seed(config.random_seed)

    out = ensure_dir(config.output_dir)
    ensure_dir(out / "figures")
    ensure_dir("data/raw")
    ensure_dir("data/processed")
    ensure_dir("logs")

    run_metadata = {
        "run_timestamp_utc": now_utc_iso(),
        "config": asdict(config),
    }

    save_json(run_metadata, out / "config_run.json")

    # Universo histórico
    membership, membership_source = load_membership_schedule(config)
    run_metadata["membership_source"] = membership_source

    tickers_needed = all_tickers_needed(
        membership,
        config.start_date,
        config.end_date,
    )

    all_download_tickers = sorted(set(tickers_needed + [config.benchmark_ticker]))

    # Descarga
    prices_raw, download_report = download_adjusted_close(
        all_download_tickers,
        config.start_date,
        config.end_date,
    )

    prices_raw.to_csv("data/raw/prices_raw.csv")
    download_report.to_csv("logs/download_report.csv", index=False)

    if config.benchmark_ticker not in prices_raw.columns:
        raise RuntimeError(
            f"No se pudo descargar el benchmark {config.benchmark_ticker}. "
            f"Prueba con '^GSPC' o revisa conexión/yfinance."
        )

    benchmark_prices = prices_raw[config.benchmark_ticker].dropna()

    asset_prices_raw = prices_raw.drop(columns=[config.benchmark_ticker], errors="ignore")
    asset_prices, exclusions = clean_prices(asset_prices_raw, config.min_price_coverage)

    asset_prices.to_csv("data/processed/prices_clean.csv")
    compute_simple_returns(asset_prices).to_csv("data/processed/returns.csv")
    exclusions.to_csv("logs/exclusions.csv", index=False)

    # Backtest de estrategia
    if config.strategy_mode == "static_buy_hold":
        strategy_returns, strategy_equity, weights, strategy_meta = backtest_static_buy_hold(
            asset_prices,
            membership,
            config,
        )

    elif config.strategy_mode == "dynamic_equal_weight":
        strategy_returns, strategy_equity, weights, strategy_meta = backtest_dynamic_equal_weight(
            asset_prices,
            membership,
            config,
        )

    else:
        raise ValueError(f"strategy_mode no reconocido: {config.strategy_mode}")

    # Benchmark
    benchmark_returns, benchmark_equity = backtest_benchmark(
        benchmark_prices,
        config,
    )

    # Alinear fechas
    common_dates = strategy_equity.index.intersection(benchmark_equity.index)

    strategy_equity = strategy_equity.loc[common_dates]
    strategy_returns = strategy_returns.loc[common_dates]

    benchmark_equity = benchmark_equity.loc[common_dates]
    benchmark_returns = benchmark_returns.loc[common_dates]

    # Métricas principales
    strategy_metrics = compute_metrics(
        strategy_returns,
        strategy_equity,
        config,
        "SP100_strategy",
    )

    benchmark_metrics = compute_metrics(
        benchmark_returns,
        benchmark_equity,
        config,
        config.benchmark_name,
    )

    metrics = pd.DataFrame([strategy_metrics, benchmark_metrics])

    # Turnover
    turnover = compute_turnover(weights)

    # Sensibilidad a costes
    sensitivity_rows = []

    for cost in config.cost_sensitivity:
        if config.strategy_mode == "static_buy_hold":
            r_s, eq_s, _, _ = backtest_static_buy_hold(
                asset_prices,
                membership,
                config,
                transaction_cost=cost,
            )

        else:
            r_s, eq_s, _, _ = backtest_dynamic_equal_weight(
                asset_prices,
                membership,
                config,
                transaction_cost=cost,
            )

        eq_s = eq_s.loc[eq_s.index.intersection(common_dates)]
        r_s = r_s.loc[eq_s.index]

        m = compute_metrics(r_s, eq_s, config, f"strategy_cost_{cost}")
        m["transaction_cost"] = cost
        sensitivity_rows.append(m)

    sensitivity = pd.DataFrame(sensitivity_rows)

    # Bootstrap simple
    boot = bootstrap_metrics_simple(
        strategy_returns,
        config,
        n_bootstrap=1000,
    )

    boot_summary = boot[["CAGR", "Sharpe", "Sortino", "max_drawdown", "Calmar"]].quantile(
        [0.025, 0.05, 0.50, 0.95, 0.975]
    )

    # Exportar outputs
    strategy_returns.to_csv(out / "strategy_returns.csv")
    strategy_equity.to_csv(out / "strategy_equity.csv")

    benchmark_returns.to_csv(out / "benchmark_returns.csv")
    benchmark_equity.to_csv(out / "benchmark_equity.csv")

    weights.to_csv(out / "weights.csv")
    turnover.to_csv(out / "turnover.csv")

    metrics.to_csv(out / "metrics.csv", index=False)
    sensitivity.to_csv(out / "cost_sensitivity.csv", index=False)

    boot.to_csv(out / "bootstrap_metrics.csv", index=False)
    boot_summary.to_csv(out / "bootstrap_summary.csv")

    # Metadata
    run_metadata["strategy_metadata"] = strategy_meta
    run_metadata["n_raw_assets_downloaded"] = len(tickers_needed)
    run_metadata["n_assets_after_cleaning"] = int(asset_prices.shape[1])
    run_metadata["n_excluded_assets"] = int(len(exclusions))
    run_metadata["benchmark_ticker"] = config.benchmark_ticker

    if "SURVIVORSHIP_BIASED" in membership_source:
        membership_warning = "WARNING: current/fallback constituents used. Survivorship bias present."
    elif "wikipedia" in membership_source.lower():
        membership_warning = (
            "Historical public reconstruction from Wikipedia page revisions. "
            "This avoids using future constituents, but is not the official licensed S&P DJI constituent history."
        )
    else:
        membership_warning = "Historical membership file used."

    run_metadata["membership_warning"] = membership_warning

    save_json(run_metadata, out / "metadata.json")

    # Figuras
    plot_equity_curves(
        strategy_equity,
        benchmark_equity,
        out / "figures/equity_curves.png",
    )

    plot_drawdowns(
        strategy_equity,
        benchmark_equity,
        out / "figures/drawdowns.png",
    )

    # Resumen por pantalla
    print("\n=== Bloque 0 completado ===")
    print(f"Modo estrategia: {config.strategy_mode}")
    print(f"Fuente membership: {membership_source}")
    print(f"Activos tras limpieza: {asset_prices.shape[1]}")
    print(f"Benchmark: {config.benchmark_ticker}")
    print(f"Outputs guardados en: {out.resolve()}")

    print("\nMétricas principales:")
    with pd.option_context("display.max_columns", None, "display.width", 160):
        print(metrics)


print("Bloque 0 cargado correctamente. Ahora cambia el Bloque 3 y ejecuta el Bloque 4.")


In [ ]:
# ============================================================
# BLOQUE 2.5 - CACHÉ DE MEMBERSHIP Y DATOS DE MERCADO


In [ ]:
# ============================================================

import hashlib


def make_market_data_cache_key(
    tickers: list,
    start_date: str,
    end_date: str,
    data_source: str,
) -> str:
    """
    Genera una clave única para identificar una descarga de datos.

    Si cambias:
    - tickers,
    - start_date,
    - end_date,
    - fuente de datos,

    se genera una caché distinta.
    """
    raw = {
        "tickers": sorted(tickers),
        "start_date": start_date,
        "end_date": end_date,
        "data_source": data_source,
    }

    raw_string = json.dumps(raw, sort_keys=True)
    return hashlib.md5(raw_string.encode("utf-8")).hexdigest()[:12]


def load_or_download_market_data(
    tickers: list,
    config: BacktestConfig,
) -> Tuple[pd.DataFrame, pd.DataFrame, str]:
    """
    Carga precios desde caché si existen.

    Si no existen o si CONFIG.force_redownload_market_data=True,
    descarga desde Yahoo Finance y guarda caché.
    """
    cache_dir = ensure_dir(getattr(config, "market_data_cache_dir", "data/cache"))

    cache_key = make_market_data_cache_key(
        tickers=tickers,
        start_date=config.start_date,
        end_date=config.end_date,
        data_source=config.data_source,
    )

    prices_cache_path = cache_dir / f"prices_{cache_key}.csv"
    report_cache_path = cache_dir / f"download_report_{cache_key}.csv"
    metadata_cache_path = cache_dir / f"market_data_metadata_{cache_key}.json"

    use_cached_market_data = getattr(config, "use_cached_market_data", True)
    force_redownload_market_data = getattr(config, "force_redownload_market_data", False)

    if (
        use_cached_market_data
        and not force_redownload_market_data
        and prices_cache_path.exists()
        and report_cache_path.exists()
    ):
        print(f"Cargando datos de mercado desde caché: {prices_cache_path}")

        prices = pd.read_csv(
            prices_cache_path,
            index_col=0,
            parse_dates=True,
        )

        prices.index.name = "Date"
        prices = prices.sort_index()

        report = pd.read_csv(report_cache_path)

        return prices, report, "market_data_cache"

    print("No hay caché válida de datos de mercado. Descargando desde Yahoo Finance...")

    prices, report = download_adjusted_close(
        tickers,
        config.start_date,
        config.end_date,
    )

    prices.to_csv(prices_cache_path)
    report.to_csv(report_cache_path, index=False)

    metadata = {
        "created_at_utc": now_utc_iso(),
        "cache_key": cache_key,
        "start_date": config.start_date,
        "end_date": config.end_date,
        "data_source": config.data_source,
        "n_tickers_requested": len(tickers),
        "tickers": sorted(tickers),
        "prices_cache_path": str(prices_cache_path),
        "report_cache_path": str(report_cache_path),
    }

    save_json(metadata, metadata_cache_path)

    print(f"Datos guardados en caché: {prices_cache_path}")

    return prices, report, "market_data_downloaded"


def run_block0(config: BacktestConfig = CONFIG) -> None:
    """
    Pipeline principal con caché.

    Cambios respecto al anterior:
    - Puede reutilizar membership histórico reconstruido.
    - Puede reutilizar precios descargados de Yahoo Finance.
    - Evita repetir los 6 minutos de descarga/reconstrucción cada vez.
    """
    warnings.filterwarnings("ignore", category=FutureWarning)
    np.random.seed(config.random_seed)

    out = ensure_dir(config.output_dir)
    ensure_dir(out / "figures")
    ensure_dir("data/raw")
    ensure_dir("data/processed")
    ensure_dir("logs")
    ensure_dir("data/cache")

    run_metadata = {
        "run_timestamp_utc": now_utc_iso(),
        "config": asdict(config),
    }

    # Guardamos también atributos añadidos dinámicamente en Colab.
    dynamic_config_attrs = [
        "use_cached_market_data",
        "force_redownload_market_data",
        "market_data_cache_dir",
    ]

    for attr in dynamic_config_attrs:
        if hasattr(config, attr):
            run_metadata["config"][attr] = getattr(config, attr)

    save_json(run_metadata, out / "config_run.json")

    # ========================================================
    # 1. Universo histórico
    # ========================================================

    membership, membership_source = load_membership_schedule(config)
    run_metadata["membership_source"] = membership_source

    tickers_needed = all_tickers_needed(
        membership,
        config.start_date,
        config.end_date,
    )

    all_download_tickers = sorted(set(tickers_needed + [config.benchmark_ticker]))

    # ========================================================
    # 2. Datos de mercado con caché
    # ========================================================

    prices_raw, download_report, market_data_source = load_or_download_market_data(
        all_download_tickers,
        config,
    )

    run_metadata["market_data_source"] = market_data_source

    prices_raw.to_csv("data/raw/prices_raw.csv")
    download_report.to_csv("logs/download_report.csv", index=False)

    if config.benchmark_ticker not in prices_raw.columns:
        raise RuntimeError(
            f"No se pudo descargar/cargar el benchmark {config.benchmark_ticker}. "
            f"Prueba con '^GSPC' o revisa conexión/yfinance."
        )

    benchmark_prices = prices_raw[config.benchmark_ticker].dropna()

    asset_prices_raw = prices_raw.drop(columns=[config.benchmark_ticker], errors="ignore")
    asset_prices, exclusions = clean_prices(asset_prices_raw, config.min_price_coverage)

    asset_prices.to_csv("data/processed/prices_clean.csv")
    compute_simple_returns(asset_prices).to_csv("data/processed/returns.csv")
    exclusions.to_csv("logs/exclusions.csv", index=False)

    # ========================================================
    # 3. Backtest de estrategia
    # ========================================================

    if config.strategy_mode == "static_buy_hold":
        strategy_returns, strategy_equity, weights, strategy_meta = backtest_static_buy_hold(
            asset_prices,
            membership,
            config,
        )

    elif config.strategy_mode == "dynamic_equal_weight":
        strategy_returns, strategy_equity, weights, strategy_meta = backtest_dynamic_equal_weight(
            asset_prices,
            membership,
            config,
        )

    else:
        raise ValueError(f"strategy_mode no reconocido: {config.strategy_mode}")

    # ========================================================
    # 4. Benchmark
    # ========================================================

    benchmark_returns, benchmark_equity = backtest_benchmark(
        benchmark_prices,
        config,
    )

    common_dates = strategy_equity.index.intersection(benchmark_equity.index)

    strategy_equity = strategy_equity.loc[common_dates]
    strategy_returns = strategy_returns.loc[common_dates]

    benchmark_equity = benchmark_equity.loc[common_dates]
    benchmark_returns = benchmark_returns.loc[common_dates]

    # ========================================================
    # 5. Métricas principales
    # ========================================================

    strategy_metrics = compute_metrics(
        strategy_returns,
        strategy_equity,
        config,
        "SP100_strategy",
    )

    benchmark_metrics = compute_metrics(
        benchmark_returns,
        benchmark_equity,
        config,
        config.benchmark_name,
    )

    metrics = pd.DataFrame([strategy_metrics, benchmark_metrics])

    # ========================================================
    # 6. Turnover
    # ========================================================

    turnover = compute_turnover(weights)

    # ========================================================
    # 7. Sensibilidad a costes
    # ========================================================

    sensitivity_rows = []

    for cost in config.cost_sensitivity:
        if config.strategy_mode == "static_buy_hold":
            r_s, eq_s, _, _ = backtest_static_buy_hold(
                asset_prices,
                membership,
                config,
                transaction_cost=cost,
            )

        else:
            r_s, eq_s, _, _ = backtest_dynamic_equal_weight(
                asset_prices,
                membership,
                config,
                transaction_cost=cost,
            )

        eq_s = eq_s.loc[eq_s.index.intersection(common_dates)]
        r_s = r_s.loc[eq_s.index]

        m = compute_metrics(r_s, eq_s, config, f"strategy_cost_{cost}")
        m["transaction_cost"] = cost
        sensitivity_rows.append(m)

    sensitivity = pd.DataFrame(sensitivity_rows)

    # ========================================================
    # 8. Bootstrap simple
    # ========================================================

    boot = bootstrap_metrics_simple(
        strategy_returns,
        config,
        n_bootstrap=1000,
    )

    boot_summary = boot[["CAGR", "Sharpe", "Sortino", "max_drawdown", "Calmar"]].quantile(
        [0.025, 0.05, 0.50, 0.95, 0.975]
    )

    # ========================================================
    # 9. Exportar outputs
    # ========================================================

    strategy_returns.to_csv(out / "strategy_returns.csv")
    strategy_equity.to_csv(out / "strategy_equity.csv")

    benchmark_returns.to_csv(out / "benchmark_returns.csv")
    benchmark_equity.to_csv(out / "benchmark_equity.csv")

    weights.to_csv(out / "weights.csv")
    turnover.to_csv(out / "turnover.csv")

    metrics.to_csv(out / "metrics.csv", index=False)
    sensitivity.to_csv(out / "cost_sensitivity.csv", index=False)

    boot.to_csv(out / "bootstrap_metrics.csv", index=False)
    boot_summary.to_csv(out / "bootstrap_summary.csv")

    # ========================================================
    # 10. Metadata
    # ========================================================

    run_metadata["strategy_metadata"] = strategy_meta
    run_metadata["n_raw_assets_downloaded"] = len(tickers_needed)
    run_metadata["n_assets_after_cleaning"] = int(asset_prices.shape[1])
    run_metadata["n_excluded_assets"] = int(len(exclusions))
    run_metadata["benchmark_ticker"] = config.benchmark_ticker

    if "SURVIVORSHIP_BIASED" in membership_source:
        membership_warning = "WARNING: current/fallback constituents used. Survivorship bias present."
    elif "wikipedia" in membership_source.lower():
        membership_warning = (
            "Historical public reconstruction from Wikipedia page revisions. "
            "This avoids using future constituents, but is not the official licensed S&P DJI constituent history."
        )
    else:
        membership_warning = "Historical membership file used."

    run_metadata["membership_warning"] = membership_warning

    save_json(run_metadata, out / "metadata.json")

    # ========================================================
    # 11. Figuras
    # ========================================================

    plot_equity_curves(
        strategy_equity,
        benchmark_equity,
        out / "figures/equity_curves.png",
    )

    plot_drawdowns(
        strategy_equity,
        benchmark_equity,
        out / "figures/drawdowns.png",
    )

    # ========================================================
    # 12. Resumen por pantalla
    # ========================================================

    print("\n=== Bloque 0 completado ===")
    print(f"Modo estrategia: {config.strategy_mode}")
    print(f"Fuente membership: {membership_source}")
    print(f"Fuente datos mercado: {market_data_source}")
    print(f"Activos tras limpieza: {asset_prices.shape[1]}")
    print(f"Benchmark: {config.benchmark_ticker}")
    print(f"Outputs guardados en: {out.resolve()}")

    print("\nMétricas principales:")
    with pd.option_context("display.max_columns", None, "display.width", 160):
        print(metrics)


print("Bloque 2.5 cargado correctamente. run_block0 ahora usa caché de datos.")

"""Configuración"""


In [ ]:
# ============================================================
# cELDA 3 - CONFIGURACIÓN DEL EXPERIMENTO


In [ ]:
# ============================================================

from pathlib import Path

# Asegurar carpetas en Drive
Path(f"{TFG_BASE_DIR}").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/data").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/data/cache").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/results").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/logs").mkdir(parents=True, exist_ok=True)

CONFIG.start_date = "2021-01-01"
CONFIG.end_date = "2026-01-01"

# Benchmark S&P 500.
CONFIG.benchmark_ticker = "SPY"

# Usamos estrategia dinámica para respetar entradas/salidas del universo histórico.
CONFIG.strategy_mode = "dynamic_equal_weight"
CONFIG.rebalance_frequency = "M"

# Costes
CONFIG.transaction_cost = 0.001

# Tasa libre de riesgo anual provisional.
CONFIG.risk_free_rate_annual = 0.0225

# Reconstrucción histórica del S&P 100.
CONFIG.use_reconstructed_membership = True

# False = no reconstruye desde Wikipedia si ya existe el CSV reconstruido.
CONFIG.force_rebuild_membership = False

CONFIG.membership_history_frequency = "MS"

# Si falla la reconstrucción histórica, NO vuelve a lista actual.
CONFIG.allow_current_fallback = False

# Caché de precios.
CONFIG.use_cached_market_data = True

# False = no vuelve a descargar precios si ya existen en caché.
CONFIG.force_redownload_market_data = False

# Guardamos caché en Drive.
CONFIG.market_data_cache_dir = f"{TFG_BASE_DIR}/data/cache"

# Guardamos resultados en Drive.
CONFIG.output_dir = f"{TFG_BASE_DIR}/results/block0_sp100_survivorship_corrected"

# Guardamos membership histórico y logs en Drive.
CONFIG.reconstructed_membership_file = f"{TFG_BASE_DIR}/data/sp100_membership_reconstructed_wikipedia.csv"
CONFIG.snapshot_log_file = f"{TFG_BASE_DIR}/data/sp100_wikipedia_snapshot_log.csv"

CONFIG

"""Ejecutar"""

run_block0(CONFIG)

"""Resultados"""


In [ ]:
# ============================================================
# CELDA 5 - VER RESULTADOS PRINCIPALES


In [ ]:
# ============================================================

from pathlib import Path
import pandas as pd
import json

results_dir = Path(CONFIG.output_dir)

metrics_path = results_dir / "metrics.csv"
sensitivity_path = results_dir / "cost_sensitivity.csv"
bootstrap_summary_path = results_dir / "bootstrap_summary.csv"
metadata_path = results_dir / "metadata.json"

print(f"Carpeta de resultados: {results_dir}")

if not metrics_path.exists():
    raise FileNotFoundError(
        f"No existe {metrics_path}. Ejecuta primero el Bloque 4: run_block0(CONFIG)"
    )

metrics = pd.read_csv(metrics_path)
cost_sensitivity = pd.read_csv(sensitivity_path)
bootstrap_summary = pd.read_csv(bootstrap_summary_path, index_col=0)

with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("\n=== Metadata clave ===")
print("Fuente membership:", metadata.get("membership_source"))
print("Aviso membership:", metadata.get("membership_warning"))
print("Fuente datos mercado:", metadata.get("market_data_source"))
print("Benchmark:", metadata.get("benchmark_ticker"))
print("Activos tras limpieza:", metadata.get("n_assets_after_cleaning"))

print("\n=== Métricas principales ===")
display(metrics)

print("\n=== Sensibilidad a costes ===")
display(cost_sensitivity)

print("\n=== Bootstrap summary ===")
display(bootstrap_summary)

"""Figuras"""


In [ ]:
# ============================================================
# CELDA 6 - VER FIGURAS


In [ ]:
# ============================================================

from pathlib import Path
from IPython.display import Image, display

results_dir = Path(CONFIG.output_dir)
figures_dir = results_dir / "figures"

equity_path = figures_dir / "equity_curves.png"
drawdown_path = figures_dir / "drawdowns.png"

print(f"Carpeta de figuras: {figures_dir}")

if not equity_path.exists():
    raise FileNotFoundError(
        f"No existe {equity_path}. Ejecuta primero el Bloque 4: run_block0(CONFIG)"
    )

if not drawdown_path.exists():
    raise FileNotFoundError(
        f"No existe {drawdown_path}. Ejecuta primero el Bloque 4: run_block0(CONFIG)"
    )

print("\n=== Curva de capital ===")
display(Image(filename=str(equity_path)))

print("\n=== Drawdown ===")
display(Image(filename=str(drawdown_path)))

"""Descargar"""

!zip -r block0_results.zip results data logs > /dev/null

from google.colab import files
files.download("block0_results.zip")


In [ ]:
# ============================================================
# CELDA C1.1 - CONFIGURACIÓN Y CARGA DE DATOS
# MÓDULO C1 - INDICADORES TÉCNICOS CLÁSICOS


In [ ]:
# ============================================================

from pathlib import Path
import json
import math
import warnings
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

# ------------------------------------------------------------
# Comprobaciones mínimas
# ------------------------------------------------------------

if "TFG_BASE_DIR" not in globals():
    TFG_BASE_DIR = "/content/drive/MyDrive/TFG"

if "CONFIG" not in globals():
    raise RuntimeError(
        "No existe CONFIG. Ejecuta primero el Módulo C0 completo."
    )

# ------------------------------------------------------------
# Fechas C1
# ------------------------------------------------------------

# Periodo real de evaluación.
C1_START_DATE = "2021-01-01"
C1_END_DATE = "2025-12-31"

# Fecha final para descarga/caché.
C1_DOWNLOAD_END = "2026-01-01"

# ------------------------------------------------------------
# Carpetas del Módulo C1
# ------------------------------------------------------------

C1_OUTPUT_DIR = Path(TFG_BASE_DIR) / "results" / "C1_technical_indicators"
C1_FIGURES_DIR = C1_OUTPUT_DIR / "figures"
C1_TABLES_DIR = C1_OUTPUT_DIR / "tables"
C1_DATA_DIR = C1_OUTPUT_DIR / "data"

for p in [C1_OUTPUT_DIR, C1_FIGURES_DIR, C1_TABLES_DIR, C1_DATA_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Parámetros del experimento
# ------------------------------------------------------------

C1_TRANSACTION_COST = CONFIG.transaction_cost
C1_COST_SENSITIVITY = CONFIG.cost_sensitivity
C1_RISK_FREE_RATE_ANNUAL = CONFIG.risk_free_rate_annual
C1_TRADING_DAYS = CONFIG.trading_days_per_year
C1_INITIAL_CAPITAL = CONFIG.initial_capital
C1_BENCHMARK_TICKER = CONFIG.benchmark_ticker
C1_MIN_OBSERVATIONS = 500

print("=== Módulo C1 configurado ===")
print("Periodo de análisis:", C1_START_DATE, "a", C1_END_DATE)
print("Fecha final descarga:", C1_DOWNLOAD_END)
print("Coste base:", C1_TRANSACTION_COST)
print("Benchmark:", C1_BENCHMARK_TICKER)
print("Carpeta de salida:", C1_OUTPUT_DIR)

# ------------------------------------------------------------
# Configuración de membership
# ------------------------------------------------------------

C1_MEMBERSHIP_CONFIG = copy.deepcopy(CONFIG)

C1_MEMBERSHIP_CONFIG.start_date = C1_START_DATE
C1_MEMBERSHIP_CONFIG.end_date = C1_DOWNLOAD_END

# Reutilizamos membership histórica de C0.
# No forzamos reconstrucción desde Wikipedia.
C1_MEMBERSHIP_CONFIG.use_reconstructed_membership = True
C1_MEMBERSHIP_CONFIG.force_rebuild_membership = False
C1_MEMBERSHIP_CONFIG.allow_current_fallback = False
C1_MEMBERSHIP_CONFIG.membership_history_frequency = "MS"

reconstructed_file = Path(C1_MEMBERSHIP_CONFIG.reconstructed_membership_file)

if not reconstructed_file.exists():
    raise FileNotFoundError(
        "No encuentro la membership reconstruida del C0.\n"
        f"Archivo esperado: {reconstructed_file}\n\n"
        "Ejecuta primero el C0 completo para generar/cargar la membership histórica."
    )

membership, membership_source = load_membership_schedule(C1_MEMBERSHIP_CONFIG)

try:
    tickers_needed = all_tickers_needed(
        membership,
        C1_START_DATE,
        C1_DOWNLOAD_END,
    )
except TypeError:
    tickers_needed = all_tickers_needed(
        membership,
        C1_MEMBERSHIP_CONFIG,
    )

benchmark_yf = to_yfinance_ticker(C1_BENCHMARK_TICKER)

tickers_for_download = sorted(set(tickers_needed + [benchmark_yf]))

# ------------------------------------------------------------
# Configuración de datos de mercado
# ------------------------------------------------------------

C1_DATA_CONFIG = copy.deepcopy(CONFIG)

C1_DATA_CONFIG.start_date = C1_START_DATE
C1_DATA_CONFIG.end_date = C1_DOWNLOAD_END
C1_DATA_CONFIG.benchmark_ticker = benchmark_yf
C1_DATA_CONFIG.transaction_cost = C1_TRANSACTION_COST
C1_DATA_CONFIG.risk_free_rate_annual = C1_RISK_FREE_RATE_ANNUAL
C1_DATA_CONFIG.output_dir = str(C1_OUTPUT_DIR)

# Reutilizamos caché común de precios.
C1_DATA_CONFIG.market_data_cache_dir = f"{TFG_BASE_DIR}/data/cache"
C1_DATA_CONFIG.use_cached_market_data = True
C1_DATA_CONFIG.force_redownload_market_data = True

raw_prices, download_report, market_data_source = load_or_download_market_data(
    tickers_for_download,
    C1_DATA_CONFIG,
)

raw_prices = raw_prices.sort_index()
raw_prices = raw_prices[~raw_prices.index.duplicated(keep="first")]

if benchmark_yf not in raw_prices.columns:
    raise RuntimeError(f"No se encuentra el benchmark {benchmark_yf} en raw_prices.")

benchmark_prices = raw_prices[benchmark_yf].dropna().copy()
asset_raw_prices = raw_prices.drop(columns=[benchmark_yf], errors="ignore").copy()

# ------------------------------------------------------------
# Limpieza por cobertura SOLO en periodo de análisis
# ------------------------------------------------------------

analysis_slice = asset_raw_prices.loc[C1_START_DATE:C1_END_DATE].copy()
coverage_analysis = analysis_slice.notna().mean()

keep_cols = coverage_analysis[
    coverage_analysis >= C1_DATA_CONFIG.min_price_coverage
].index.tolist()

excluded_cols = coverage_analysis[
    coverage_analysis < C1_DATA_CONFIG.min_price_coverage
].index.tolist()

prices_clean = asset_raw_prices[keep_cols].copy()
prices_clean = prices_clean.ffill()

# Cortamos al periodo real de análisis.
prices_clean = prices_clean.loc[C1_START_DATE:C1_END_DATE].copy()
prices_clean = prices_clean.dropna(axis=1, how="all")

# ------------------------------------------------------------
# Benchmark calculado igual que C4/C5
# ------------------------------------------------------------

benchmark_returns = benchmark_prices.pct_change(fill_method=None)
benchmark_returns = benchmark_returns.replace([np.inf, -np.inf], np.nan).fillna(0.0)
benchmark_returns.name = "Benchmark"

benchmark_returns = benchmark_returns.loc[C1_START_DATE:C1_END_DATE].copy()

# Alineamos activos y benchmark al mismo calendario.
common_index = prices_clean.index.intersection(benchmark_returns.index)

prices_clean = prices_clean.loc[common_index].copy()
benchmark_returns = benchmark_returns.loc[common_index].fillna(0.0)

benchmark_equity = (1.0 + benchmark_returns).cumprod()
benchmark_equity.name = "Benchmark"

# ------------------------------------------------------------
# Diagnósticos y guardado
# ------------------------------------------------------------

download_report.to_csv(C1_DATA_DIR / "C1_download_report.csv", index=False)

exclusions = pd.DataFrame({
    "yf_ticker": excluded_cols,
    "reason": f"coverage_analysis_below_{C1_DATA_CONFIG.min_price_coverage:.0%}",
})

if len(excluded_cols) > 0:
    exclusions["coverage_analysis"] = coverage_analysis.loc[excluded_cols].values

exclusions.to_csv(C1_TABLES_DIR / "C1_exclusions.csv", index=False)

prices_clean.to_csv(C1_DATA_DIR / "C1_prices_clean.csv")
benchmark_returns.to_csv(C1_DATA_DIR / "C1_benchmark_returns.csv")
benchmark_equity.to_csv(C1_DATA_DIR / "C1_benchmark_equity.csv")

print("\n=== Carga completada ===")
print("Membership source:", membership_source)
print("Market data source:", market_data_source)
print("Membership reconstruida:", reconstructed_file)
print("Forzar reconstrucción membership:", C1_MEMBERSHIP_CONFIG.force_rebuild_membership)
print("Tickers solicitados:", len(tickers_for_download))
print("Activos tras limpieza:", prices_clean.shape[1])
print("Activos excluidos:", len(excluded_cols))
print("Fechas disponibles:", prices_clean.shape[0])
print("Fechas:", prices_clean.index.min(), "->", prices_clean.index.max())

print("\nBenchmark calculado desde precios:")
print("Inicio:", benchmark_returns.index.min())
print("Fin:", benchmark_returns.index.max())
print("Retorno acumulado benchmark:", f"{100 * (benchmark_equity.iloc[-1] / benchmark_equity.iloc[0] - 1):.2f}%")

display(prices_clean.head())
display(prices_clean.tail())


In [ ]:
# ============================================================
# CELDA C1.2 - FUNCIONES DE MÉTRICAS, SEÑALES Y BACKTEST


In [ ]:
# ============================================================

def c1_drawdown_series(equity: pd.Series) -> pd.Series:
    equity = pd.Series(equity).dropna()

    if len(equity) == 0:
        return pd.Series(dtype=float)

    running_max = equity.cummax()
    return equity / running_max - 1.0


def c1_max_drawdown(equity: pd.Series) -> float:
    dd = c1_drawdown_series(equity)

    if len(dd) == 0:
        return np.nan

    return float(dd.min())


def c1_cagr(equity: pd.Series) -> float:
    equity = pd.Series(equity).dropna()

    if len(equity) < 2:
        return np.nan

    years = len(equity) / C1_TRADING_DAYS

    if years <= 0:
        return np.nan

    final_value = equity.iloc[-1]

    if final_value <= 0:
        return -1.0

    return float((final_value / equity.iloc[0]) ** (1.0 / years) - 1.0)


def c1_compute_metrics(
    returns: pd.Series,
    equity: pd.Series,
    label: str,
    ticker: str | None = None,
    strategy: str | None = None,
) -> dict:
    returns = pd.Series(returns).replace([np.inf, -np.inf], np.nan).dropna()
    equity = pd.Series(equity).replace([np.inf, -np.inf], np.nan).dropna()

    if len(returns) < 2 or len(equity) < 2:
        return {
            "ticker": ticker,
            "strategy": strategy,
            "label": label,
            "start": None,
            "end": None,
            "n_observations": len(returns),
            "cumulative_return": np.nan,
            "CAGR": np.nan,
            "annualized_volatility": np.nan,
            "Sharpe": np.nan,
            "Sortino": np.nan,
            "max_drawdown": np.nan,
            "Calmar": np.nan,
            "hit_ratio": np.nan,
            "risk_free_rate_annual": C1_RISK_FREE_RATE_ANNUAL,
        }

    rf_daily = (1.0 + C1_RISK_FREE_RATE_ANNUAL) ** (1.0 / C1_TRADING_DAYS) - 1.0
    excess = returns - rf_daily

    cumulative_return = float(equity.iloc[-1] / equity.iloc[0] - 1.0)
    cagr = c1_cagr(equity)

    annualized_volatility = float(returns.std(ddof=1) * math.sqrt(C1_TRADING_DAYS))

    if excess.std(ddof=1) > 0:
        sharpe = float(excess.mean() / excess.std(ddof=1) * math.sqrt(C1_TRADING_DAYS))
    else:
        sharpe = np.nan

    downside = excess[excess < 0]

    if len(downside) > 1 and downside.std(ddof=1) > 0:
        sortino = float(excess.mean() / downside.std(ddof=1) * math.sqrt(C1_TRADING_DAYS))
    else:
        sortino = np.nan

    mdd = c1_max_drawdown(equity)

    if not np.isnan(mdd) and mdd < 0:
        calmar = float(cagr / abs(mdd))
    else:
        calmar = np.nan

    hit_ratio = float((returns > 0).mean())

    return {
        "ticker": ticker,
        "strategy": strategy,
        "label": label,
        "start": str(returns.index[0].date()) if hasattr(returns.index[0], "date") else str(returns.index[0]),
        "end": str(returns.index[-1].date()) if hasattr(returns.index[-1], "date") else str(returns.index[-1]),
        "n_observations": int(len(returns)),
        "cumulative_return": cumulative_return,
        "CAGR": cagr,
        "annualized_volatility": annualized_volatility,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "max_drawdown": mdd,
        "Calmar": calmar,
        "hit_ratio": hit_ratio,
        "risk_free_rate_annual": C1_RISK_FREE_RATE_ANNUAL,
    }


def c1_build_active_mask(prices: pd.DataFrame, membership: pd.DataFrame | None) -> pd.DataFrame:
    """
    Construye una máscara booleana fecha-activo.
    True significa que el activo pertenece al universo histórico en esa fecha.
    """
    if membership is None:
        return prices.notna()

    mask = pd.DataFrame(False, index=prices.index, columns=prices.columns)

    for row in membership.itertuples(index=False):
        ticker = getattr(row, "yf_ticker", None)

        if ticker not in prices.columns:
            continue

        start = getattr(row, "start", pd.NaT)
        end = getattr(row, "end", pd.NaT)

        if pd.isna(start):
            start = prices.index.min()

        if pd.isna(end):
            end = prices.index.max()

        date_mask = (prices.index >= start) & (prices.index <= end)
        mask.loc[date_mask, ticker] = True

    return mask & prices.notna()


def c1_macd_position(prices: pd.Series, fast: int = 12, slow: int = 26) -> pd.Series:
    """
    Regla tipo MACD usada en el TFG:
    - long si EMA rápida > EMA lenta
    - short si EMA rápida < EMA lenta
    """
    prices = pd.Series(prices).dropna()

    ema_fast = prices.ewm(span=fast, adjust=False, min_periods=fast).mean()
    ema_slow = prices.ewm(span=slow, adjust=False, min_periods=slow).mean()

    position = pd.Series(np.nan, index=prices.index)
    position[ema_fast > ema_slow] = 1.0
    position[ema_fast < ema_slow] = -1.0

    position = position.ffill().fillna(0.0)
    position.name = "MACD"

    return position


def c1_rsi_position(
    prices: pd.Series,
    window: int = 14,
    low: float = 30,
    high: float = 70,
) -> pd.Series:
    """
    Regla RSI:
    - long si RSI < 30
    - short si RSI > 70
    - mantiene posición entre medias
    """
    prices = pd.Series(prices).dropna()

    delta = prices.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)

    avg_gain = gain.rolling(window=window, min_periods=window).mean()
    avg_loss = loss.rolling(window=window, min_periods=window).mean()

    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    rsi = 100.0 - (100.0 / (1.0 + rs))

    # Si avg_loss=0 y avg_gain>0, RSI tiende a 100.
    rsi = rsi.fillna(100.0).where(avg_gain.notna() & avg_loss.notna(), np.nan)

    signal = pd.Series(np.nan, index=prices.index)
    signal[rsi < low] = 1.0
    signal[rsi > high] = -1.0

    position = signal.ffill().fillna(0.0)
    position.name = "RSI"

    return position


def c1_stochastic_position(
    prices: pd.Series,
    window: int = 14,
    low: float = 20,
    high: float = 80,
) -> pd.Series:
    """
    Regla oscilador estocástico:
    - long si %K < 20
    - short si %K > 80
    - mantiene posición entre medias
    """
    prices = pd.Series(prices).dropna()

    rolling_low = prices.rolling(window=window, min_periods=window).min()
    rolling_high = prices.rolling(window=window, min_periods=window).max()

    denominator = rolling_high - rolling_low
    k_percent = 100.0 * (prices - rolling_low) / denominator.replace(0.0, np.nan)

    signal = pd.Series(np.nan, index=prices.index)
    signal[k_percent < low] = 1.0
    signal[k_percent > high] = -1.0

    position = signal.ffill().fillna(0.0)
    position.name = "Oscilador estocástico"

    return position


def c1_backtest_single_asset(
    prices: pd.Series,
    position: pd.Series,
    transaction_cost: float,
) -> tuple[pd.Series, pd.Series, pd.Series]:
    """
    Backtest de un único activo.
    La posición calculada en t se aplica sobre la rentabilidad de t+1.
    """
    prices = pd.Series(prices).dropna()
    position = pd.Series(position).reindex(prices.index).ffill().fillna(0.0)

    asset_returns = prices.pct_change(fill_method=None)

    position_lagged = position.shift(1).fillna(0.0)
    turnover = position_lagged.diff().abs()
    turnover = turnover.fillna(position_lagged.abs())

    strategy_returns = position_lagged * asset_returns - transaction_cost * turnover
    strategy_returns = strategy_returns.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    strategy_returns.name = "strategy_returns"

    equity = (1.0 + strategy_returns).cumprod()
    equity.name = "equity"

    return strategy_returns, equity, position_lagged


def c1_buyhold_single_asset(prices: pd.Series) -> tuple[pd.Series, pd.Series]:
    prices = pd.Series(prices).dropna()

    returns = prices.pct_change(fill_method=None).fillna(0.0)
    returns.name = "buyhold_returns"

    equity = (1.0 + returns).cumprod()
    equity.name = "buyhold_equity"

    return returns, equity


def c1_aggregate_equal_weight(returns_df: pd.DataFrame) -> tuple[pd.Series, pd.Series]:
    """
    Cartera agregada equally-weighted a partir de las rentabilidades
    disponibles por activo en cada fecha.
    """
    returns_df = returns_df.replace([np.inf, -np.inf], np.nan)

    aggregate_returns = returns_df.mean(axis=1, skipna=True).fillna(0.0)
    aggregate_returns.name = "aggregate_returns"

    aggregate_equity = (1.0 + aggregate_returns).cumprod()
    aggregate_equity.name = "aggregate_equity"

    return aggregate_returns, aggregate_equity


C1_STRATEGY_FUNCTIONS = {
    "MACD": c1_macd_position,
    "RSI": c1_rsi_position,
    "Oscilador estocástico": c1_stochastic_position,
}

print("Funciones del Módulo C1 cargadas correctamente.")


In [ ]:
# ============================================================
# CELDA C1.2 - FUNCIONES DE MÉTRICAS, SEÑALES Y BACKTEST


In [ ]:
# ============================================================

def c1_equity_from_returns(returns: pd.Series) -> pd.Series:
    """
    Construye la curva de capital a partir de retornos.

    Importante:
    El primer valor de la serie ya incluye el primer retorno.
    Por tanto, el capital final correcto es equity.iloc[-1],
    no equity.iloc[-1] / equity.iloc[0].
    """
    returns = pd.Series(returns).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    equity = (1.0 + returns).cumprod()
    equity.name = "equity"
    return equity


def c1_drawdown_series(equity: pd.Series) -> pd.Series:
    """
    Calcula drawdown incluyendo capital inicial 1 antes del primer retorno.

    Esto evita perder el drawdown del primer día si el primer retorno es negativo.
    """
    equity = pd.Series(equity).replace([np.inf, -np.inf], np.nan).dropna()

    if len(equity) == 0:
        return pd.Series(dtype=float)

    # Añadimos capital inicial 1 justo antes de la primera observación.
    # No importa demasiado el índice porque solo usamos la serie para drawdown.
    equity_with_initial = pd.concat([
        pd.Series([1.0], index=[equity.index[0] - pd.Timedelta(days=1)]),
        equity,
    ])

    running_max = equity_with_initial.cummax()
    dd = equity_with_initial / running_max - 1.0

    # Devolvemos solo fechas reales.
    return dd.iloc[1:]


def c1_max_drawdown(equity: pd.Series) -> float:
    dd = c1_drawdown_series(equity)

    if len(dd) == 0:
        return np.nan

    return float(dd.min())


def c1_cagr_from_returns(returns: pd.Series) -> float:
    """
    CAGR calculada directamente desde retornos.

    Esto incluye correctamente el primer retorno del periodo.
    """
    returns = pd.Series(returns).replace([np.inf, -np.inf], np.nan).dropna()

    if len(returns) < 2:
        return np.nan

    cumulative_growth = float((1.0 + returns).prod())

    if cumulative_growth <= 0:
        return -1.0

    years = len(returns) / C1_TRADING_DAYS

    if years <= 0:
        return np.nan

    return float(cumulative_growth ** (1.0 / years) - 1.0)


def c1_compute_metrics(
    returns: pd.Series,
    equity: pd.Series | None,
    label: str,
    ticker: str | None = None,
    strategy: str | None = None,
) -> dict:
    """
    Calcula métricas usando retornos como fuente principal.

    Motivo:
    Si la curva de capital empieza ya después del primer retorno,
    usar equity.iloc[-1] / equity.iloc[0] elimina el primer día.
    Por eso la rentabilidad acumulada y la CAGR se calculan desde returns.
    """
    returns = pd.Series(returns).replace([np.inf, -np.inf], np.nan).dropna()

    if len(returns) < 2:
        return {
            "ticker": ticker,
            "strategy": strategy,
            "label": label,
            "start": None,
            "end": None,
            "n_observations": len(returns),
            "cumulative_return": np.nan,
            "CAGR": np.nan,
            "annualized_volatility": np.nan,
            "Sharpe": np.nan,
            "Sortino": np.nan,
            "max_drawdown": np.nan,
            "Calmar": np.nan,
            "hit_ratio": np.nan,
            "risk_free_rate_annual": C1_RISK_FREE_RATE_ANNUAL,
        }

    # Reconstruimos equity desde returns para que todas las métricas sean coherentes.
    equity = c1_equity_from_returns(returns)

    rf_daily = (1.0 + C1_RISK_FREE_RATE_ANNUAL) ** (1.0 / C1_TRADING_DAYS) - 1.0
    excess = returns - rf_daily

    cumulative_return = float((1.0 + returns).prod() - 1.0)
    cagr = c1_cagr_from_returns(returns)

    annualized_volatility = float(returns.std(ddof=1) * math.sqrt(C1_TRADING_DAYS))

    if excess.std(ddof=1) > 0:
        sharpe = float(excess.mean() / excess.std(ddof=1) * math.sqrt(C1_TRADING_DAYS))
    else:
        sharpe = np.nan

    downside = excess[excess < 0]

    if len(downside) > 1 and downside.std(ddof=1) > 0:
        sortino = float(excess.mean() / downside.std(ddof=1) * math.sqrt(C1_TRADING_DAYS))
    else:
        sortino = np.nan

    mdd = c1_max_drawdown(equity)

    if not np.isnan(mdd) and mdd < 0:
        calmar = float(cagr / abs(mdd))
    else:
        calmar = np.nan

    hit_ratio = float((returns > 0).mean())

    return {
        "ticker": ticker,
        "strategy": strategy,
        "label": label,
        "start": str(returns.index[0].date()) if hasattr(returns.index[0], "date") else str(returns.index[0]),
        "end": str(returns.index[-1].date()) if hasattr(returns.index[-1], "date") else str(returns.index[-1]),
        "n_observations": int(len(returns)),
        "cumulative_return": cumulative_return,
        "CAGR": cagr,
        "annualized_volatility": annualized_volatility,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "max_drawdown": mdd,
        "Calmar": calmar,
        "hit_ratio": hit_ratio,
        "risk_free_rate_annual": C1_RISK_FREE_RATE_ANNUAL,
    }


def c1_build_active_mask(prices: pd.DataFrame, membership: pd.DataFrame | None) -> pd.DataFrame:
    """
    Construye una máscara booleana fecha-activo.
    True significa que el activo pertenece al universo histórico en esa fecha.
    """
    if membership is None:
        return prices.notna()

    mask = pd.DataFrame(False, index=prices.index, columns=prices.columns)

    for row in membership.itertuples(index=False):
        ticker = getattr(row, "yf_ticker", None)

        if ticker not in prices.columns:
            continue

        start = getattr(row, "start", pd.NaT)
        end = getattr(row, "end", pd.NaT)

        if pd.isna(start):
            start = prices.index.min()

        if pd.isna(end):
            end = prices.index.max()

        date_mask = (prices.index >= start) & (prices.index <= end)
        mask.loc[date_mask, ticker] = True

    return mask & prices.notna()


def c1_macd_position(prices: pd.Series, fast: int = 12, slow: int = 26) -> pd.Series:
    """
    Regla tipo MACD usada en el TFG:
    - long si EMA rápida > EMA lenta
    - short si EMA rápida < EMA lenta
    """
    prices = pd.Series(prices).dropna()

    ema_fast = prices.ewm(span=fast, adjust=False, min_periods=fast).mean()
    ema_slow = prices.ewm(span=slow, adjust=False, min_periods=slow).mean()

    position = pd.Series(np.nan, index=prices.index)
    position[ema_fast > ema_slow] = 1.0
    position[ema_fast < ema_slow] = -1.0

    position = position.ffill().fillna(0.0)
    position.name = "MACD"

    return position


def c1_rsi_position(
    prices: pd.Series,
    window: int = 14,
    low: float = 30,
    high: float = 70,
) -> pd.Series:
    """
    Regla RSI:
    - long si RSI < 30
    - short si RSI > 70
    - mantiene posición entre medias
    """
    prices = pd.Series(prices).dropna()

    delta = prices.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)

    avg_gain = gain.rolling(window=window, min_periods=window).mean()
    avg_loss = loss.rolling(window=window, min_periods=window).mean()

    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    rsi = 100.0 - (100.0 / (1.0 + rs))

    # Si avg_loss=0 y avg_gain>0, RSI tiende a 100.
    rsi = rsi.fillna(100.0).where(avg_gain.notna() & avg_loss.notna(), np.nan)

    signal = pd.Series(np.nan, index=prices.index)
    signal[rsi < low] = 1.0
    signal[rsi > high] = -1.0

    position = signal.ffill().fillna(0.0)
    position.name = "RSI"

    return position


def c1_stochastic_position(
    prices: pd.Series,
    window: int = 14,
    low: float = 20,
    high: float = 80,
) -> pd.Series:
    """
    Regla oscilador estocástico:
    - long si %K < 20
    - short si %K > 80
    - mantiene posición entre medias
    """
    prices = pd.Series(prices).dropna()

    rolling_low = prices.rolling(window=window, min_periods=window).min()
    rolling_high = prices.rolling(window=window, min_periods=window).max()

    denominator = rolling_high - rolling_low
    k_percent = 100.0 * (prices - rolling_low) / denominator.replace(0.0, np.nan)

    signal = pd.Series(np.nan, index=prices.index)
    signal[k_percent < low] = 1.0
    signal[k_percent > high] = -1.0

    position = signal.ffill().fillna(0.0)
    position.name = "Oscilador estocástico"

    return position


def c1_backtest_single_asset(
    prices: pd.Series,
    position: pd.Series,
    transaction_cost: float,
) -> tuple[pd.Series, pd.Series, pd.Series]:
    """
    Backtest de un único activo.
    La posición calculada en t se aplica sobre la rentabilidad de t+1.
    """
    prices = pd.Series(prices).dropna()
    position = pd.Series(position).reindex(prices.index).ffill().fillna(0.0)

    asset_returns = prices.pct_change(fill_method=None)

    position_lagged = position.shift(1).fillna(0.0)
    turnover = position_lagged.diff().abs()
    turnover = turnover.fillna(position_lagged.abs())

    strategy_returns = position_lagged * asset_returns - transaction_cost * turnover
    strategy_returns = strategy_returns.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    strategy_returns.name = "strategy_returns"

    equity = c1_equity_from_returns(strategy_returns)
    equity.name = "equity"

    return strategy_returns, equity, position_lagged


def c1_buyhold_single_asset(prices: pd.Series) -> tuple[pd.Series, pd.Series]:
    prices = pd.Series(prices).dropna()

    returns = prices.pct_change(fill_method=None).fillna(0.0)
    returns.name = "buyhold_returns"

    equity = c1_equity_from_returns(returns)
    equity.name = "buyhold_equity"

    return returns, equity


def c1_aggregate_equal_weight(returns_df: pd.DataFrame) -> tuple[pd.Series, pd.Series]:
    """
    Cartera agregada equally-weighted a partir de las rentabilidades
    disponibles por activo en cada fecha.
    """
    returns_df = returns_df.replace([np.inf, -np.inf], np.nan)

    aggregate_returns = returns_df.mean(axis=1, skipna=True).fillna(0.0)
    aggregate_returns.name = "aggregate_returns"

    aggregate_equity = c1_equity_from_returns(aggregate_returns)
    aggregate_equity.name = "aggregate_equity"

    return aggregate_returns, aggregate_equity


C1_STRATEGY_FUNCTIONS = {
    "MACD": c1_macd_position,
    "RSI": c1_rsi_position,
    "Oscilador estocástico": c1_stochastic_position,
}

print("Funciones del Módulo C1 cargadas correctamente.")


In [ ]:
# ============================================================
# CELDA C1.3 - EJECUTAR BACKTESTS POR ACTIVO


In [ ]:
# ============================================================

# ------------------------------------------------------------
# Reconstruir índice de análisis si no existe
# ------------------------------------------------------------

if "C1_ANALYSIS_START" not in globals():
    C1_ANALYSIS_START = C1_START_DATE

if "C1_ANALYSIS_END" not in globals():
    C1_ANALYSIS_END = C1_END_DATE

if "C1_ANALYSIS_INDEX" not in globals():
    C1_ANALYSIS_INDEX = prices_clean.index[
        (prices_clean.index >= pd.Timestamp(C1_ANALYSIS_START)) &
        (prices_clean.index <= pd.Timestamp(C1_ANALYSIS_END))
    ]

    if "benchmark_returns" in globals():
        C1_ANALYSIS_INDEX = C1_ANALYSIS_INDEX.intersection(benchmark_returns.index)

    if len(C1_ANALYSIS_INDEX) == 0:
        raise RuntimeError("No hay fechas disponibles en el periodo de análisis C1.")

    C1_VALID_START = C1_ANALYSIS_INDEX.min()
    C1_VALID_END = C1_ANALYSIS_INDEX.max()

else:
    C1_VALID_START = C1_ANALYSIS_INDEX.min()
    C1_VALID_END = C1_ANALYSIS_INDEX.max()

analysis_index = C1_ANALYSIS_INDEX

print("Periodo análisis C1:", C1_VALID_START.date(), "->", C1_VALID_END.date())
print("Observaciones análisis:", len(analysis_index))

# ------------------------------------------------------------
# Máscara de activos activos según membership histórica
# ------------------------------------------------------------

active_mask = c1_build_active_mask(prices_clean, membership)

individual_rows = []

returns_by_strategy = {
    name: pd.DataFrame(index=analysis_index)
    for name in C1_STRATEGY_FUNCTIONS
}

positions_by_strategy = {
    name: pd.DataFrame(index=analysis_index)
    for name in C1_STRATEGY_FUNCTIONS
}

buyhold_returns_by_asset = pd.DataFrame(index=analysis_index)

valid_tickers = []
skipped_tickers = []

# ------------------------------------------------------------
# Backtests individuales
# ------------------------------------------------------------

for idx, ticker in enumerate(prices_clean.columns, start=1):

    # Usamos toda la serie disponible desde 2020 para calcular indicadores.
    full_prices = prices_clean[ticker].dropna()

    if len(full_prices) < C1_MIN_OBSERVATIONS:
        skipped_tickers.append({
            "ticker": ticker,
            "reason": f"less_than_{C1_MIN_OBSERVATIONS}_total_observations",
            "n_observations": len(full_prices),
        })
        continue

    # Solo evaluamos cuando el activo pertenece al universo durante el periodo análisis.
    eval_mask = active_mask[ticker].reindex(prices_clean.index).fillna(False)
    eval_mask = eval_mask & prices_clean.index.isin(analysis_index)

    eval_prices = prices_clean[ticker].where(eval_mask).dropna()

    if len(eval_prices) < C1_MIN_OBSERVATIONS:
        skipped_tickers.append({
            "ticker": ticker,
            "reason": f"less_than_{C1_MIN_OBSERVATIONS}_analysis_observations",
            "n_observations": len(eval_prices),
        })
        continue

    valid_tickers.append(ticker)

    # --------------------------------------------------------
    # Buy & Hold por activo
    # --------------------------------------------------------

    bh_returns_full, _ = c1_buyhold_single_asset(full_prices)

    bh_returns_eval = bh_returns_full.reindex(prices_clean.index)
    bh_returns_eval = bh_returns_eval.where(eval_mask)
    bh_returns_eval = bh_returns_eval.loc[analysis_index].dropna()

    bh_equity_eval = (1.0 + bh_returns_eval).cumprod()
    bh_equity_eval.name = "buyhold_equity"

    buyhold_returns_by_asset[ticker] = bh_returns_eval.reindex(analysis_index)

    bh_metrics = c1_compute_metrics(
        returns=bh_returns_eval,
        equity=bh_equity_eval,
        label=f"{ticker}_BuyHold",
        ticker=ticker,
        strategy="Buy&Hold",
    )

    individual_rows.append(bh_metrics)

    # --------------------------------------------------------
    # Estrategias técnicas
    # --------------------------------------------------------

    for strategy_name, strategy_func in C1_STRATEGY_FUNCTIONS.items():
        try:
            # La señal se calcula con histórico previo.
            position_full = strategy_func(full_prices)

            strategy_returns_full, strategy_equity_full, position_lagged_full = c1_backtest_single_asset(
                prices=full_prices,
                position=position_full,
                transaction_cost=C1_TRANSACTION_COST,
            )

            strategy_returns_eval = strategy_returns_full.reindex(prices_clean.index)
            strategy_returns_eval = strategy_returns_eval.where(eval_mask)
            strategy_returns_eval = strategy_returns_eval.loc[analysis_index].dropna()

            position_lagged_eval = position_lagged_full.reindex(prices_clean.index)
            position_lagged_eval = position_lagged_eval.where(eval_mask)
            position_lagged_eval = position_lagged_eval.loc[analysis_index]

            strategy_equity_eval = (1.0 + strategy_returns_eval).cumprod()
            strategy_equity_eval.name = "strategy_equity"

            returns_by_strategy[strategy_name][ticker] = strategy_returns_eval.reindex(analysis_index)
            positions_by_strategy[strategy_name][ticker] = position_lagged_eval.reindex(analysis_index)

            metrics_row = c1_compute_metrics(
                returns=strategy_returns_eval,
                equity=strategy_equity_eval,
                label=f"{ticker}_{strategy_name}",
                ticker=ticker,
                strategy=strategy_name,
            )

            individual_rows.append(metrics_row)

        except Exception as e:
            skipped_tickers.append({
                "ticker": ticker,
                "reason": f"{strategy_name}_error_{type(e).__name__}: {e}",
                "n_observations": len(eval_prices),
            })

    if idx % 25 == 0:
        print(f"Procesados {idx} activos de {prices_clean.shape[1]}...")

# ------------------------------------------------------------
# Resultados finales
# ------------------------------------------------------------

individual_results = pd.DataFrame(individual_rows)
skipped_tickers = pd.DataFrame(skipped_tickers)

if individual_results.empty:
    raise RuntimeError("No se ha generado ningún resultado individual en C1.3.")

individual_results = individual_results[
    [
        "ticker",
        "strategy",
        "label",
        "start",
        "end",
        "n_observations",
        "cumulative_return",
        "CAGR",
        "annualized_volatility",
        "Sharpe",
        "Sortino",
        "max_drawdown",
        "Calmar",
        "hit_ratio",
        "risk_free_rate_annual",
    ]
]

print("\n=== Backtests individuales completados ===")
print("Periodo análisis:", C1_VALID_START.date(), "->", C1_VALID_END.date())
print("Activos válidos:", len(valid_tickers))
print("Activos omitidos:", len(skipped_tickers))
print("Filas de resultados:", len(individual_results))

display(individual_results.head())

if len(skipped_tickers) > 0:
    print("\nActivos omitidos:")
    display(skipped_tickers.head())


In [ ]:
# ============================================================
# CELDA C1.4 - RESULTADOS AGREGADOS Y SENSIBILIDAD A COSTES


In [ ]:
# ============================================================

# ------------------------------------------------------------
# Fallbacks defensivos
# ------------------------------------------------------------

if "C1_DATA_START" not in globals():
    C1_DATA_START = "2020-01-01"

if "C1_ANALYSIS_START" not in globals():
    if "C1_START_DATE" in globals():
        C1_ANALYSIS_START = C1_START_DATE
    else:
        C1_ANALYSIS_START = "2021-01-01"

if "C1_ANALYSIS_END" not in globals():
    if "C1_END_DATE" in globals():
        C1_ANALYSIS_END = C1_END_DATE
    else:
        C1_ANALYSIS_END = "2025-12-31"

if "C1_ANALYSIS_INDEX" not in globals():
    C1_ANALYSIS_INDEX = prices_clean.index[
        (prices_clean.index >= pd.Timestamp(C1_ANALYSIS_START)) &
        (prices_clean.index <= pd.Timestamp(C1_ANALYSIS_END))
    ]

analysis_index = C1_ANALYSIS_INDEX

if len(analysis_index) == 0:
    raise RuntimeError("No hay fechas de análisis disponibles en C1.")

C1_VALID_START = analysis_index.min()
C1_VALID_END = analysis_index.max()

print("Periodo análisis C1:", C1_VALID_START.date(), "->", C1_VALID_END.date())
print("Observaciones análisis:", len(analysis_index))

# ------------------------------------------------------------
# Media y mediana por estrategia
# ------------------------------------------------------------

summary_mean = (
    individual_results
    .groupby("strategy")
    .agg(
        n_assets=("ticker", "count"),
        cumulative_return_mean=("cumulative_return", "mean"),
        CAGR_mean=("CAGR", "mean"),
        annualized_volatility_mean=("annualized_volatility", "mean"),
        Sharpe_mean=("Sharpe", "mean"),
        Sortino_mean=("Sortino", "mean"),
        max_drawdown_mean=("max_drawdown", "mean"),
        Calmar_mean=("Calmar", "mean"),
        hit_ratio_mean=("hit_ratio", "mean"),
    )
    .reset_index()
)

summary_median = (
    individual_results
    .groupby("strategy")
    .agg(
        n_assets=("ticker", "count"),
        cumulative_return_median=("cumulative_return", "median"),
        CAGR_median=("CAGR", "median"),
        annualized_volatility_median=("annualized_volatility", "median"),
        Sharpe_median=("Sharpe", "median"),
        Sortino_median=("Sortino", "median"),
        max_drawdown_median=("max_drawdown", "median"),
        Calmar_median=("Calmar", "median"),
        hit_ratio_median=("hit_ratio", "median"),
    )
    .reset_index()
)

strategy_order = ["MACD", "RSI", "Oscilador estocástico", "Buy&Hold"]

summary_mean["strategy"] = pd.Categorical(
    summary_mean["strategy"],
    categories=strategy_order,
    ordered=True,
)

summary_median["strategy"] = pd.Categorical(
    summary_median["strategy"],
    categories=strategy_order,
    ordered=True,
)

summary_mean = summary_mean.sort_values("strategy").reset_index(drop=True)
summary_median = summary_median.sort_values("strategy").reset_index(drop=True)

print("=== Media por estrategia ===")
display(summary_mean)

print("=== Mediana por estrategia ===")
display(summary_median)

# ------------------------------------------------------------
# Cartera agregada equal-weight por estrategia
# ------------------------------------------------------------

aggregate_returns = pd.DataFrame(index=analysis_index)
aggregate_equity = pd.DataFrame(index=analysis_index)

for strategy_name, strategy_returns_df in returns_by_strategy.items():
    r_agg, eq_agg = c1_aggregate_equal_weight(strategy_returns_df)

    r_agg = r_agg.reindex(analysis_index).fillna(0.0)
    eq_agg = (1.0 + r_agg).cumprod()

    aggregate_returns[strategy_name] = r_agg
    aggregate_equity[strategy_name] = eq_agg

buyhold_aggregate_returns, buyhold_aggregate_equity = c1_aggregate_equal_weight(
    buyhold_returns_by_asset
)

buyhold_aggregate_returns = buyhold_aggregate_returns.reindex(analysis_index).fillna(0.0)
buyhold_aggregate_equity = (1.0 + buyhold_aggregate_returns).cumprod()

aggregate_returns["Buy&Hold"] = buyhold_aggregate_returns
aggregate_equity["Buy&Hold"] = buyhold_aggregate_equity

# ------------------------------------------------------------
# Benchmark SPY robusto
# ------------------------------------------------------------

def c1_get_benchmark_returns_robust(analysis_index):
    """
    Devuelve los retornos de SPY alineados al periodo de análisis.

    Prioridad:
    1. Usar benchmark_prices si contiene precio previo a 2021.
    2. Usar raw_prices si contiene SPY con precio previo.
    3. Descargar SPY directamente con yfinance desde 2020.
    4. Si todo falla, usar benchmark_returns existente, avisando.
    """

    first_analysis_date = analysis_index[0]
    last_analysis_date = analysis_index[-1]

    benchmark_symbol = "SPY"

    if "C1_BENCHMARK_TICKER" in globals():
        benchmark_symbol = C1_BENCHMARK_TICKER

    if "to_yfinance_ticker" in globals():
        benchmark_symbol_yf = to_yfinance_ticker(benchmark_symbol)
    else:
        benchmark_symbol_yf = benchmark_symbol

    candidate_prices = None
    source = None

    # 1. benchmark_prices
    if "benchmark_prices" in globals():
        tmp = pd.Series(benchmark_prices).sort_index().dropna()

        if len(tmp.loc[tmp.index < first_analysis_date]) > 0:
            candidate_prices = tmp.copy()
            source = "benchmark_prices"

    # 2. raw_prices
    if candidate_prices is None and "raw_prices" in globals():
        if benchmark_symbol_yf in raw_prices.columns:
            tmp = raw_prices[benchmark_symbol_yf].sort_index().dropna()

            if len(tmp.loc[tmp.index < first_analysis_date]) > 0:
                candidate_prices = tmp.copy()
                source = "raw_prices"

    # 3. yfinance directo
    if candidate_prices is None:
        try:
            import yfinance as yf

            downloaded = yf.download(
                benchmark_symbol_yf,
                start=C1_DATA_START,
                end="2026-01-01",
                auto_adjust=True,
                progress=False,
            )

            if isinstance(downloaded.columns, pd.MultiIndex):
                downloaded.columns = downloaded.columns.get_level_values(0)

            if "Close" in downloaded.columns:
                tmp = downloaded["Close"].sort_index().dropna()
            elif "Adj Close" in downloaded.columns:
                tmp = downloaded["Adj Close"].sort_index().dropna()
            else:
                tmp = downloaded.iloc[:, 0].sort_index().dropna()

            if len(tmp.loc[tmp.index < first_analysis_date]) > 0:
                candidate_prices = tmp.copy()
                source = "yfinance_direct"

        except Exception as e:
            print("No se pudo descargar benchmark por yfinance:", e)

    # 4. fallback final
    if candidate_prices is None:
        print("AVISO: no he encontrado precio previo de SPY.")
        print("Uso benchmark_returns existente. Puede salir 98,08% en vez de 95,39%.")

        if "benchmark_returns" not in globals():
            raise RuntimeError("No existe benchmark_returns y no se pudo reconstruir SPY.")

        out = pd.Series(benchmark_returns).reindex(analysis_index).fillna(0.0)
        out.name = "Benchmark"
        return out, "fallback_existing_benchmark_returns"

    # Calcular retornos desde precios con precio previo.
    candidate_prices = candidate_prices.loc[
        candidate_prices.index <= last_analysis_date
    ].copy()

    full_returns = candidate_prices.pct_change(fill_method=None)
    full_returns = full_returns.replace([np.inf, -np.inf], np.nan)

    out = full_returns.reindex(analysis_index)

    # Forzar primer retorno contra último cierre anterior.
    if pd.isna(out.iloc[0]):
        previous_prices = candidate_prices.loc[candidate_prices.index < first_analysis_date]

        if len(previous_prices) == 0:
            raise RuntimeError(
                "No hay precio previo al inicio de análisis incluso tras reconstruir SPY."
            )

        prev_price = previous_prices.iloc[-1]
        first_price = candidate_prices.loc[first_analysis_date]

        out.iloc[0] = first_price / prev_price - 1.0

    out = out.fillna(0.0)
    out.name = "Benchmark"

    return out, source


benchmark_returns_aligned, benchmark_source_c1 = c1_get_benchmark_returns_robust(
    analysis_index
)

benchmark_equity_aligned = (1.0 + benchmark_returns_aligned).cumprod()

aggregate_returns["Benchmark"] = benchmark_returns_aligned
aggregate_equity["Benchmark"] = benchmark_equity_aligned

print("\nBenchmark C1 recalculado:")
print("Fuente:", benchmark_source_c1)
print("Primer día:", benchmark_returns_aligned.index[0].date())
print("Primer retorno:", f"{100 * benchmark_returns_aligned.iloc[0]:.4f}%")
print("Retorno acumulado:", f"{100 * ((1.0 + benchmark_returns_aligned).prod() - 1.0):.2f}%")

# ------------------------------------------------------------
# Métricas agregadas
# ------------------------------------------------------------

aggregate_metrics_rows = []

for col in aggregate_returns.columns:
    aggregate_metrics_rows.append(
        c1_compute_metrics(
            returns=aggregate_returns[col],
            equity=aggregate_equity[col],
            label=col,
            ticker=None,
            strategy=col,
        )
    )

aggregate_metrics = pd.DataFrame(aggregate_metrics_rows)

aggregate_metrics = aggregate_metrics[
    [
        "strategy",
        "label",
        "start",
        "end",
        "n_observations",
        "cumulative_return",
        "CAGR",
        "annualized_volatility",
        "Sharpe",
        "Sortino",
        "max_drawdown",
        "Calmar",
        "hit_ratio",
        "risk_free_rate_annual",
    ]
]

print("=== Métricas agregadas ===")
display(aggregate_metrics)

# ------------------------------------------------------------
# Distribución de posiciones
# ------------------------------------------------------------

position_distribution_rows = []

for strategy_name, positions_df in positions_by_strategy.items():
    values = positions_df.stack().dropna()

    if len(values) == 0:
        continue

    position_distribution_rows.append({
        "strategy": strategy_name,
        "n_position_observations": int(len(values)),
        "long_ratio": float((values > 0).mean()),
        "short_ratio": float((values < 0).mean()),
        "flat_ratio": float((values == 0).mean()),
        "average_abs_position": float(values.abs().mean()),
    })

position_distribution = pd.DataFrame(position_distribution_rows)

print("=== Distribución de posiciones ===")
display(position_distribution)

# ------------------------------------------------------------
# Sensibilidad a costes
# ------------------------------------------------------------

cost_sensitivity_rows = []

for cost in C1_COST_SENSITIVITY:
    print(f"Calculando sensibilidad a coste = {cost}")

    cost_returns_by_strategy = {
        name: pd.DataFrame(index=analysis_index)
        for name in C1_STRATEGY_FUNCTIONS
    }

    for ticker in valid_tickers:

        full_prices = prices_clean[ticker].dropna()

        eval_mask = active_mask[ticker].reindex(prices_clean.index).fillna(False)
        eval_mask = eval_mask & prices_clean.index.isin(analysis_index)

        eval_prices = prices_clean[ticker].where(eval_mask).dropna()

        if len(eval_prices) < C1_MIN_OBSERVATIONS:
            continue

        for strategy_name, strategy_func in C1_STRATEGY_FUNCTIONS.items():
            position_full = strategy_func(full_prices)

            strategy_returns_full, strategy_equity_full, position_lagged_full = c1_backtest_single_asset(
                prices=full_prices,
                position=position_full,
                transaction_cost=cost,
            )

            strategy_returns_eval = strategy_returns_full.reindex(prices_clean.index)
            strategy_returns_eval = strategy_returns_eval.where(eval_mask)
            strategy_returns_eval = strategy_returns_eval.loc[analysis_index].dropna()

            cost_returns_by_strategy[strategy_name][ticker] = strategy_returns_eval.reindex(analysis_index)

    for strategy_name, strategy_returns_df in cost_returns_by_strategy.items():
        r_agg, eq_agg = c1_aggregate_equal_weight(strategy_returns_df)

        r_agg = r_agg.reindex(analysis_index).fillna(0.0)
        eq_agg = (1.0 + r_agg).cumprod()

        row = c1_compute_metrics(
            returns=r_agg,
            equity=eq_agg,
            label=f"{strategy_name}_cost_{cost}",
            strategy=strategy_name,
        )

        row["transaction_cost"] = cost
        cost_sensitivity_rows.append(row)

cost_sensitivity = pd.DataFrame(cost_sensitivity_rows)

cost_sensitivity = cost_sensitivity[
    [
        "strategy",
        "transaction_cost",
        "n_observations",
        "cumulative_return",
        "CAGR",
        "annualized_volatility",
        "Sharpe",
        "Sortino",
        "max_drawdown",
        "Calmar",
        "hit_ratio",
    ]
]

print("=== Sensibilidad a costes ===")
display(cost_sensitivity)


In [ ]:
# ============================================================
# CELDA C1.5 - GUARDAR TABLAS Y VERSIONES LATEX


In [ ]:
# ============================================================

# ------------------------------------------------------------
# Fallbacks defensivos de fechas y configuración
# ------------------------------------------------------------

if "C1_DATA_START" not in globals():
    # Si esta variable no existe, intentamos inferirla.
    # Lo ideal es que sea 2020-01-01 para usar calentamiento.
    C1_DATA_START = "2020-01-01"

if "C1_ANALYSIS_START" not in globals():
    if "C1_START_DATE" in globals():
        C1_ANALYSIS_START = C1_START_DATE
    else:
        C1_ANALYSIS_START = "2021-01-01"

if "C1_ANALYSIS_END" not in globals():
    if "C1_END_DATE" in globals():
        C1_ANALYSIS_END = C1_END_DATE
    else:
        C1_ANALYSIS_END = "2025-12-31"

if "C1_VALID_START" not in globals():
    if "analysis_index" in globals() and len(analysis_index) > 0:
        C1_VALID_START = analysis_index.min()
    elif "C1_ANALYSIS_INDEX" in globals() and len(C1_ANALYSIS_INDEX) > 0:
        C1_VALID_START = C1_ANALYSIS_INDEX.min()
    else:
        C1_VALID_START = pd.Timestamp(C1_ANALYSIS_START)

if "C1_VALID_END" not in globals():
    if "analysis_index" in globals() and len(analysis_index) > 0:
        C1_VALID_END = analysis_index.max()
    elif "C1_ANALYSIS_INDEX" in globals() and len(C1_ANALYSIS_INDEX) > 0:
        C1_VALID_END = C1_ANALYSIS_INDEX.max()
    else:
        C1_VALID_END = pd.Timestamp(C1_ANALYSIS_END)

if "analysis_index" not in globals():
    if "C1_ANALYSIS_INDEX" in globals():
        analysis_index = C1_ANALYSIS_INDEX
    else:
        analysis_index = prices_clean.index[
            (prices_clean.index >= pd.Timestamp(C1_ANALYSIS_START)) &
            (prices_clean.index <= pd.Timestamp(C1_ANALYSIS_END))
        ]

if "C1_COST_SENSITIVITY" not in globals():
    C1_COST_SENSITIVITY = CONFIG.cost_sensitivity

if "C1_TRANSACTION_COST" not in globals():
    C1_TRANSACTION_COST = CONFIG.transaction_cost

if "C1_RISK_FREE_RATE_ANNUAL" not in globals():
    C1_RISK_FREE_RATE_ANNUAL = CONFIG.risk_free_rate_annual

if "C1_MIN_OBSERVATIONS" not in globals():
    C1_MIN_OBSERVATIONS = 500

if "C1_BENCHMARK_TICKER" not in globals():
    C1_BENCHMARK_TICKER = CONFIG.benchmark_ticker

if "membership_source" not in globals():
    membership_source = "not_available"

if "market_data_source" not in globals():
    market_data_source = "not_available"

# ------------------------------------------------------------
# Función auxiliar
# ------------------------------------------------------------

def c1_format_percent_table(df: pd.DataFrame, percent_cols: list[str]) -> pd.DataFrame:
    out = df.copy()

    for col in percent_cols:
        if col in out.columns:
            out[col] = 100.0 * out[col]

    return out


# ------------------------------------------------------------
# Comprobaciones mínimas antes de guardar
# ------------------------------------------------------------

required_objects = [
    "individual_results",
    "summary_mean",
    "summary_median",
    "aggregate_metrics",
    "aggregate_returns",
    "aggregate_equity",
    "cost_sensitivity",
    "position_distribution",
    "skipped_tickers",
]

missing_objects = [obj for obj in required_objects if obj not in globals()]

if len(missing_objects) > 0:
    raise RuntimeError(
        "Faltan objetos necesarios para C1.5. "
        "Ejecuta antes C1.3 y C1.4.\n"
        f"Faltan: {missing_objects}"
    )

# ------------------------------------------------------------
# Guardar CSV
# ------------------------------------------------------------

individual_results.to_csv(C1_TABLES_DIR / "C1_individual_results.csv", index=False)
summary_mean.to_csv(C1_TABLES_DIR / "C1_summary_mean.csv", index=False)
summary_median.to_csv(C1_TABLES_DIR / "C1_summary_median.csv", index=False)
aggregate_metrics.to_csv(C1_TABLES_DIR / "C1_aggregate_metrics.csv", index=False)

aggregate_returns.to_csv(C1_OUTPUT_DIR / "C1_aggregate_returns.csv")
aggregate_equity.to_csv(C1_OUTPUT_DIR / "C1_aggregate_equity.csv")

cost_sensitivity.to_csv(C1_TABLES_DIR / "C1_cost_sensitivity.csv", index=False)
position_distribution.to_csv(C1_TABLES_DIR / "C1_position_distribution.csv", index=False)
skipped_tickers.to_csv(C1_TABLES_DIR / "C1_skipped_tickers.csv", index=False)

# ------------------------------------------------------------
# Tablas preparadas para el TFG
# ------------------------------------------------------------

summary_mean_for_latex = c1_format_percent_table(
    summary_mean,
    [
        "cumulative_return_mean",
        "CAGR_mean",
        "annualized_volatility_mean",
        "max_drawdown_mean",
        "hit_ratio_mean",
    ],
)

summary_mean_for_latex = summary_mean_for_latex.rename(columns={
    "strategy": "Estrategia",
    "n_assets": "N",
    "cumulative_return_mean": "Rent. acum. media (\\%)",
    "CAGR_mean": "CAGR media (\\%)",
    "annualized_volatility_mean": "Vol. media (\\%)",
    "Sharpe_mean": "Sharpe medio",
    "Sortino_mean": "Sortino medio",
    "max_drawdown_mean": "MDD medio (\\%)",
    "Calmar_mean": "Calmar medio",
    "hit_ratio_mean": "Hit ratio medio (\\%)",
})

summary_median_for_latex = c1_format_percent_table(
    summary_median,
    [
        "cumulative_return_median",
        "CAGR_median",
        "annualized_volatility_median",
        "max_drawdown_median",
        "hit_ratio_median",
    ],
)

summary_median_for_latex = summary_median_for_latex.rename(columns={
    "strategy": "Estrategia",
    "n_assets": "N",
    "cumulative_return_median": "Rent. acum. mediana (\\%)",
    "CAGR_median": "CAGR mediana (\\%)",
    "annualized_volatility_median": "Vol. mediana (\\%)",
    "Sharpe_median": "Sharpe mediano",
    "Sortino_median": "Sortino mediano",
    "max_drawdown_median": "MDD mediano (\\%)",
    "Calmar_median": "Calmar mediano",
    "hit_ratio_median": "Hit ratio mediano (\\%)",
})

aggregate_metrics_for_latex = c1_format_percent_table(
    aggregate_metrics,
    [
        "cumulative_return",
        "CAGR",
        "annualized_volatility",
        "max_drawdown",
        "hit_ratio",
    ],
)

aggregate_metrics_for_latex = aggregate_metrics_for_latex.rename(columns={
    "strategy": "Estrategia",
    "n_observations": "N obs.",
    "cumulative_return": "Rent. acum. (\\%)",
    "CAGR": "CAGR (\\%)",
    "annualized_volatility": "Vol. (\\%)",
    "Sharpe": "Sharpe",
    "Sortino": "Sortino",
    "max_drawdown": "MDD (\\%)",
    "Calmar": "Calmar",
    "hit_ratio": "Hit ratio (\\%)",
})

# Columnas finales compactas para el TFG
summary_mean_for_latex = summary_mean_for_latex[
    [
        "Estrategia",
        "N",
        "Rent. acum. media (\\%)",
        "CAGR media (\\%)",
        "Vol. media (\\%)",
        "Sharpe medio",
        "MDD medio (\\%)",
        "Calmar medio",
    ]
]

summary_median_for_latex = summary_median_for_latex[
    [
        "Estrategia",
        "N",
        "Rent. acum. mediana (\\%)",
        "CAGR mediana (\\%)",
        "Vol. mediana (\\%)",
        "Sharpe mediano",
        "MDD mediano (\\%)",
        "Calmar mediano",
    ]
]

aggregate_metrics_for_latex = aggregate_metrics_for_latex[
    [
        "Estrategia",
        "N obs.",
        "Rent. acum. (\\%)",
        "CAGR (\\%)",
        "Vol. (\\%)",
        "Sharpe",
        "MDD (\\%)",
        "Calmar",
    ]
]

with open(C1_TABLES_DIR / "C1_table_summary_mean.tex", "w", encoding="utf-8") as f:
    f.write(summary_mean_for_latex.to_latex(index=False, float_format="%.3f", escape=False))

with open(C1_TABLES_DIR / "C1_table_summary_median.tex", "w", encoding="utf-8") as f:
    f.write(summary_median_for_latex.to_latex(index=False, float_format="%.3f", escape=False))

with open(C1_TABLES_DIR / "C1_table_aggregate_metrics.tex", "w", encoding="utf-8") as f:
    f.write(aggregate_metrics_for_latex.to_latex(index=False, float_format="%.3f", escape=False))

# ------------------------------------------------------------
# Metadata del experimento
# ------------------------------------------------------------

C1_metadata = {
    "module": "C1_technical_indicators",
    "description": (
        "Indicadores técnicos clásicos: MACD, RSI y oscilador estocástico. "
        "Se usan datos previos a 2021 solo como calentamiento para indicadores "
        "y primer retorno; la evaluación real es 2021-2025."
    ),
    "data_start": C1_DATA_START,
    "analysis_start": C1_ANALYSIS_START,
    "analysis_end": C1_ANALYSIS_END,
    "valid_start": str(pd.Timestamp(C1_VALID_START).date()),
    "valid_end": str(pd.Timestamp(C1_VALID_END).date()),
    "universe": "S&P 100 historical reconstruction from C0",
    "membership_source": membership_source,
    "market_data_source": market_data_source,
    "benchmark_ticker": C1_BENCHMARK_TICKER,
    "transaction_cost": C1_TRANSACTION_COST,
    "cost_sensitivity": list(C1_COST_SENSITIVITY),
    "risk_free_rate_annual": C1_RISK_FREE_RATE_ANNUAL,
    "min_observations_per_asset": C1_MIN_OBSERVATIONS,
    "n_valid_tickers": len(valid_tickers) if "valid_tickers" in globals() else None,
    "n_skipped_tickers": int(len(skipped_tickers)),
    "n_observations_analysis": int(len(analysis_index)),
    "strategies": {
        "MACD": "Long if EMA12 > EMA26, short if EMA12 < EMA26.",
        "RSI": "Long if RSI < 30, short if RSI > 70, otherwise keep previous position.",
        "Oscilador estocástico": "Long if %K < 20, short if %K > 80, otherwise keep previous position.",
    },
    "output_dir": str(C1_OUTPUT_DIR),
}

with open(C1_OUTPUT_DIR / "C1_metadata.json", "w", encoding="utf-8") as f:
    json.dump(C1_metadata, f, indent=4, ensure_ascii=False)

print("=== Tablas guardadas ===")
print(C1_TABLES_DIR)

print("\nFechas metadata:")
print("data_start:", C1_DATA_START)
print("analysis_start:", C1_ANALYSIS_START)
print("analysis_end:", C1_ANALYSIS_END)
print("valid_start:", C1_VALID_START)
print("valid_end:", C1_VALID_END)

print("\nTabla media para TFG:")
display(summary_mean_for_latex)

print("\nTabla mediana para TFG:")
display(summary_median_for_latex)

print("\nTabla agregada para TFG:")
display(aggregate_metrics_for_latex)


In [ ]:
# ============================================================
# CELDA C1.6 - GENERAR GRÁFICAS


In [ ]:
# ============================================================

def c1_save_equity_curves(equity_df: pd.DataFrame, output_path: Path) -> None:
    plt.figure(figsize=(10, 6))

    for col in equity_df.columns:
        equity_df[col].dropna().plot(label=col)

    plt.title("Módulo C1: curvas de capital agregadas")
    plt.xlabel("Fecha")
    plt.ylabel("Capital normalizado")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()


def c1_save_drawdowns(equity_df: pd.DataFrame, output_path: Path) -> None:
    plt.figure(figsize=(10, 6))

    for col in equity_df.columns:
        c1_drawdown_series(equity_df[col]).dropna().plot(label=col)

    plt.title("Módulo C1: drawdowns agregados")
    plt.xlabel("Fecha")
    plt.ylabel("Drawdown")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()


def c1_save_metric_bar(metrics_df: pd.DataFrame, metric: str, title: str, output_path: Path) -> None:
    plot_df = metrics_df.copy()
    plot_df = plot_df.set_index("strategy")

    plt.figure(figsize=(9, 5))
    plot_df[metric].plot(kind="bar")
    plt.title(title)
    plt.xlabel("Estrategia")
    plt.ylabel(metric)
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()


def c1_save_boxplot(individual_df: pd.DataFrame, metric: str, title: str, output_path: Path) -> None:
    strategies = ["MACD", "RSI", "Oscilador estocástico", "Buy&Hold"]
    data = []

    for strategy in strategies:
        values = individual_df[individual_df["strategy"] == strategy][metric].dropna()
        data.append(values)

    plt.figure(figsize=(9, 5))
    plt.boxplot(data, labels=strategies, showfliers=False)
    plt.title(title)
    plt.xlabel("Estrategia")
    plt.ylabel(metric)
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()


def c1_save_cost_sensitivity_plot(cost_df: pd.DataFrame, output_path: Path) -> None:
    plt.figure(figsize=(9, 5))

    for strategy in cost_df["strategy"].dropna().unique():
        sub = cost_df[cost_df["strategy"] == strategy].sort_values("transaction_cost")
        plt.plot(sub["transaction_cost"], sub["Sharpe"], marker="o", label=strategy)

    plt.title("Sensibilidad del Sharpe a costes de transacción")
    plt.xlabel("Coste proporcional")
    plt.ylabel("Sharpe")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()


def c1_save_position_distribution_plot(position_df: pd.DataFrame, output_path: Path) -> None:
    plot_df = position_df.set_index("strategy")[["long_ratio", "short_ratio", "flat_ratio"]]

    plt.figure(figsize=(9, 5))
    plot_df.plot(kind="bar", stacked=True)
    plt.title("Distribución de posiciones por estrategia")
    plt.xlabel("Estrategia")
    plt.ylabel("Proporción")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()


# ------------------------------------------------------------
# Crear figuras
# ------------------------------------------------------------

c1_save_equity_curves(
    aggregate_equity,
    C1_FIGURES_DIR / "C1_equity_curves_aggregate.png",
)

c1_save_drawdowns(
    aggregate_equity,
    C1_FIGURES_DIR / "C1_drawdowns_aggregate.png",
)

c1_save_metric_bar(
    aggregate_metrics,
    "CAGR",
    "CAGR agregada por estrategia",
    C1_FIGURES_DIR / "C1_bar_CAGR_aggregate.png",
)

c1_save_metric_bar(
    aggregate_metrics,
    "Sharpe",
    "Sharpe agregado por estrategia",
    C1_FIGURES_DIR / "C1_bar_Sharpe_aggregate.png",
)

c1_save_metric_bar(
    aggregate_metrics,
    "max_drawdown",
    "Máximo drawdown agregado por estrategia",
    C1_FIGURES_DIR / "C1_bar_MDD_aggregate.png",
)

c1_save_boxplot(
    individual_results,
    "Sharpe",
    "Distribución del Sharpe por activo",
    C1_FIGURES_DIR / "C1_boxplot_Sharpe_by_asset.png",
)

c1_save_boxplot(
    individual_results,
    "CAGR",
    "Distribución de la CAGR por activo",
    C1_FIGURES_DIR / "C1_boxplot_CAGR_by_asset.png",
)

c1_save_boxplot(
    individual_results,
    "max_drawdown",
    "Distribución del máximo drawdown por activo",
    C1_FIGURES_DIR / "C1_boxplot_MDD_by_asset.png",
)

c1_save_cost_sensitivity_plot(
    cost_sensitivity,
    C1_FIGURES_DIR / "C1_cost_sensitivity_Sharpe.png",
)

if len(position_distribution) > 0:
    c1_save_position_distribution_plot(
        position_distribution,
        C1_FIGURES_DIR / "C1_position_distribution.png",
    )

print("=== Figuras guardadas ===")
print(C1_FIGURES_DIR)

for p in sorted(C1_FIGURES_DIR.glob("*.png")):
    print("-", p.name)


In [ ]:
# ============================================================
# CELDA C1.7 - VER RESULTADOS Y FIGURAS


In [ ]:
# ============================================================

print("=== Resumen medio por estrategia ===")
display(summary_mean)

print("=== Resumen mediano por estrategia ===")
display(summary_median)

print("=== Métricas agregadas ===")
display(aggregate_metrics)

print("=== Sensibilidad a costes ===")
display(cost_sensitivity)

print("=== Distribución de posiciones ===")
display(position_distribution)

figure_paths = [
    C1_FIGURES_DIR / "C1_equity_curves_aggregate.png",
    C1_FIGURES_DIR / "C1_drawdowns_aggregate.png",
    C1_FIGURES_DIR / "C1_bar_CAGR_aggregate.png",
    C1_FIGURES_DIR / "C1_bar_Sharpe_aggregate.png",
    C1_FIGURES_DIR / "C1_bar_MDD_aggregate.png",
    C1_FIGURES_DIR / "C1_boxplot_Sharpe_by_asset.png",
    C1_FIGURES_DIR / "C1_boxplot_CAGR_by_asset.png",
    C1_FIGURES_DIR / "C1_boxplot_MDD_by_asset.png",
    C1_FIGURES_DIR / "C1_cost_sensitivity_Sharpe.png",
    C1_FIGURES_DIR / "C1_position_distribution.png",
]

for fig_path in figure_paths:
    if fig_path.exists():
        print(f"\n=== {fig_path.name} ===")
        display(Image(filename=str(fig_path)))


In [ ]:
# ============================================================
# CELDA C1.8 - COMPRIMIR Y DESCARGAR RESULTADOS DEL MÓDULO C1


In [ ]:
# ============================================================

from google.colab import files

zip_name = "C1_technical_indicators_results.zip"

if not C1_OUTPUT_DIR.exists():
    raise FileNotFoundError(
        f"No existe {C1_OUTPUT_DIR}. Ejecuta primero el Módulo C1."
    )

!zip -r "{zip_name}" "{C1_OUTPUT_DIR}" > /dev/null

files.download(zip_name)
